
# 搭建LLM基本模型
## 总体结构

每个模块（第 2 到 8 章）统一用同一个四段式：**数学 → 实现 → 对拍 staff → 代价与 kernel 归属**。得到 Part 1 需要的"代码到 kernel 对照表"。

```
0. 目标、约定与环境
1. 记号与形状约定
2. Embedding
3. Linear
4. RMSNorm
5. RoPE
6. 因果多头自注意力
7. SwiGLU 前馈
8. Transformer Block（pre-norm 残差）
9. 整体拼装：BasicsTransformerLM
10. 与 staff 实现的整体验证
11. 代价核算（FLOPs / 显存 / 访存）
12. 代码 → kernel 对照表与 NVTX 标注点
13. 附录：推导与自测题
```


## 各章内容

### 0. 目标、约定与环境
- 本 notebook 的目标：理解 Part 1 要测的对象，产出 `cs336_systems/mini_model.py`
- 验收标准：`load_state_dict` 通过，logits `allclose`，梯度也 `allclose`
- 环境：用 `uv run` 启动 Jupyter（需要 `uv add --dev ipykernel jupyter`），固定随机种子，统一 fp32 且 TF32 关

### 1. 记号与形状约定
#### 符号表

| 符号 | 含义 | 代码里的名字 | 作业取值 / 关系 | 注解 |
|---|---|---|---|---|
| B | batch size | 输入 `x` 的第 0 维 | 作业 benchmark 固定为 4 | 一次并行处理的序列条数 |
| T | 序列长度（context length） | `context_length`、`sequence_length` | 128 / 256 / 512 / 1024 | 注意力的代价按 T² 增长 |
| N | 一个 batch 的 token 总数 | 无 | N = B·T | 线性层的 FLOPs 只和 N 成正比，和 B、T 的拆分无关 |
| d | 模型宽度 | `d_model` | small 768、medium 1024、large 1280、xl 1600、2.7B 2560 | 残差流的通道数，几乎所有张量的最后一维 |
| h | 注意力头数 | `num_heads` | 12 / 16 / 20 / 25 / 32 | 必须整除 d |
| d_k | 每个头的维度 | `d_k`（`d_v` 与之相等） | d_k = d / h，前四档都是 64，2.7B 为 80 | h·d_k = d，拆头不增加参数量 |
| d_ff | FFN 的内层宽度 | `d_ff` | d_ff = 4d（3072 / 4096 / 5120 / 6400 / 10240） | SwiGLU 有 3 个 d×d_ff 的矩阵 |
| V | 词表大小 | `vocab_size` | 10000 | 决定 embedding 与 lm_head 的大小，也决定 logits 的显存 |
| L | Transformer 层数 | `num_layers` | 12 / 24 / 36 / 48 / 32 | 深度，参数与 FLOPs 都按 L 线性增长 |

补充约定：
- **与 Step 2 的区别**：Step 2 的公式里用 `T = B·ctx` 表示 token 总数。notebook 里 T 已经表示序列长度，所以**token 总数改用 N = B·T**，Step 2 的公式里把 T 换成 N 即可，不要混用。
- 张量形状里用 `...` 表示任意前导维度（与 staff 源码的 jaxtyping 注解一致）。
- 作业不绑定 embedding 与 lm_head，二者各占 V·d 个参数。
- 全模型的张量形状流：`(B,T)` → `(B,T,d)` → `(B,h,T,d_k)` → `(B,T,V)`
- 约定一次，后面每章的形状注释都用这套

```python
"I love deep learning"          ← 原始文本
   ↓ ① 分词器切分
["I", " love", " deep", " learning"]   ← tokens（子词/字符/单词，取决于分词器）
   ↓ ② 查词表：每个 token 字符串 → 一个固定整数
[4, 5, 6, 7]                    ← token_ids
   ↓ ③ Embedding 查表
(5, d_model) 浮点矩阵            ← 进入 Transformer
   ...
   ↓ ④ 模型末端的 LM 头：d_model → vocab_size 个分数
softmax / 采样 → 得到一个 id
   ↓ ⑤ 反查词表 id → token 字符串 → 拼回文本
```

In [3]:
from __future__ import annotations

import json
import logging
import math
import os
import warnings

import einx
import torch
import torch.nn as nn
from einops import einsum, rearrange
from jaxtyping import Bool, Float, Int
from torch import Tensor


### Linear


### 2. Embedding

**2.1 数学**
- 设权重 $W\in\mathbb{R}^{V\times d}$，第 $i$ 行是词 $i$ 的向量。对 id $x$：$E(x)=W_{x,:}$。
- 形状规则：ids 为任意形状 `(...)`，输出为 `(..., d)`。
- 与 one-hot 的关系：令 $O\in\{0,1\}^{N\times V}$，$O_{n,x_n}=1$，则 $Y=OW$。查表就是这个矩阵乘的结果，但**不真的构造 $O$**。
- 代价对比：one-hot 矩阵乘需要 $2NVd$ FLOPs（与 lm_head 同量级），查表是 0 FLOPs，只读 $N\cdot d$ 个元素。

**2.2 可微：梯度是 scatter-add**
- 前向是 gather，反向是 scatter-add：
  $$\frac{\partial L}{\partial W_{i,:}}=\sum_{n:\,x_n=i}\frac{\partial L}{\partial Y_{n,:}}$$
- 三个结论：
  1. 本 batch 没出现的词，对应行的梯度为 0；
  2. 重复出现的词，梯度累加（演示用的 `[0,3,2,2]` 里，词 2 的梯度就是两份之和）；
  3. id 是整数，自身没有梯度。
- 系统视角：梯度张量仍是**稠密的 $V\times d$**（大部分行为 0），AdamW 也会更新所有行，这部分显存开销不随 batch 变化。

**2.3 实现与初始化**
- 实现：`self.weight[token_ids, :]`（高级索引），对照 `F.embedding`。
- 初始化：截断正态，std=1，截断区间 $[-3,3]$（作业规定）。
- 注意：原论文在 embedding 之后会乘 $\sqrt{d_{model}}$，staff 源码**没有乘**，代码注释里已说明；本 notebook 也不乘。

**2.4 对拍**
- 演示单元（代码，输入与输出分开展示）：
  1. $V=5,\ d=4$ 的小权重，`weight[2]`、`weight[[0,3,2,2]]`；
  2. ids 形状 `(2,3)` → 输出 `(2,3,4)`；
  3. 三种实现的等价性：手写查表、`F.embedding`、one-hot 矩阵乘。
- 断言：
  - 手写查表与 `F.embedding`：`torch.equal`；
  - one-hot 矩阵乘：`torch.allclose`；
  - 与 staff 的 `Embedding`，同权重：`torch.equal`。
- 边界：越界 id 在 CPU 上报 `IndexError`，在 GPU 上会触发 device-side assert，调试时先在 CPU 上复现。

**2.5 代价与 kernel 归属**
- 参数量：$V\cdot d$；FLOPs：0；访存：读写各约 $N\cdot d$ 个元素，量级很小。
- 预期 kernel：前向是 gather / index 类；反向是 scatter-add（`index_put` 或 embedding backward 类）。具体名字以 nsys 为准，回填到第 12 章对照表。
- 预期占比：对训练步时间影响很小，但它的稠密梯度和 Adam 状态占用 $V\cdot d$ 的显存。



In [57]:
class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        std = 1.0
        self.weight = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(vocab_size, d_model), std=std, a=-3 * std, b=3 * std), requires_grad=True
            )
    def forward(self,token_ids:Int[Tensor,"..."])->Float[Tensor,"..., d_model"]:
        return self.weight[token_ids,:]


### 3. Linear
- 数学：`y = x Wᵀ`，权重形状为 `(d_out, d_in)` 的原因
- 初始化推导：std = √(2/(d_in+d_out))（Xavier 形式），截断在 ±3σ
- 实现：先写 `x @ W.T`，再解释 staff 用 einsum 的写法，二者等价
- 代价：2·T·d_in·d_out FLOPs，是计算密集型
- 与 Part 1 的关联：einsum 在底层是否额外产生 permute 或 copy，留给 nsys 验证

In [76]:
class Linear(nn.Module):
    def __init__(self,d_in:int,d_out:int):
        # d_in -> d_out
        # initialized with truncated normal fan-in fan-out.
        super().__init__()
        std = math.sqrt(2 / (d_in + d_out))
        self.weight = nn.Parameter(
            nn.init.trunc_normal_(torch.empty(d_out, d_in), std=std, a=-3 * std, b=3 * std), requires_grad=True
        )

    def forward(
            self,
            x:Float[Tensor,"... d_in"]
        )->Float[Tensor,"... d_out"]:
        return einsum(x,self.weight,"... d_in, d_out d_in-> ... d_out")

### 4. RMSNorm

#### 4.1 数学：定义

符号：对每个 token 的 $d$ 维向量 $x\in\mathbb{R}^d$ 独立归一化（$d$ 为隐藏维，不是 token 总数 $N=B\cdot T$）。

$$
r(x)=\Big(\tfrac1d\textstyle\sum_{j=1}^d x_j^2+\epsilon\Big)^{-1/2},\qquad
\hat x = r\cdot x,\qquad
y = w\odot \hat x+b
$$

- $w\in\mathbb{R}^d$ 是可学习的逐通道缩放，初始化为 1。作业（staff 实现）**没有 bias**，$\epsilon=10^{-5}$，且 $\epsilon$ 在根号**里面**。
- 含义：只做**重缩放**，让每个向量的均方根变为 1：$\frac1d\sum\hat x_j^2=1$（$\epsilon\to0$ 时）。不做中心化。

**与 LayerNorm 的对比**

$$
\mu=\tfrac1d\textstyle\sum x_j,\quad \sigma^2=\tfrac1d\textstyle\sum (x_j-\mu)^2,\quad
\hat x=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}},\quad y=w\odot\hat x+b
$$

| | LayerNorm | RMSNorm |
|---|---|---|
| 统计量 | 均值 $\mu$ 和方差 $\sigma^2$ | 只有 $\frac1d\sum x^2$ |
| 中心化 | 有（减 $\mu$） | 无 |
| 可学习参数 | $w$ 和 $b$ | 只有 $w$ |
| 遍历次数 | 可用 $\sum x,\sum x^2$ 一次遍历算出，**不一定多一次扫描** | 一次 |

结论：RMSNorm 省的是一个统计量、一步减法和 $b$ 的参数与梯度，**不是**扫描次数。在访存受限的情况下，两者的性能差异主要看实现是否融合。

#### 4.2 数学：反向传播

记 $g=\delta_y\odot w$（对 $\hat x$ 的梯度），$\delta_y=\partial L/\partial y$。

**推导**（逐分量）：$\hat x_j=r\,x_j$，且 $\partial r/\partial x_i=-\,r^3x_i/d$，所以

$$
\frac{\partial L}{\partial x_i}=\sum_j g_j\frac{\partial \hat x_j}{\partial x_i}
= r\,g_i-\frac{r^3x_i}{d}\sum_j g_jx_j .
$$

利用 $r^3x_ix_j=r\,\hat x_i\hat x_j$，得到

$$
\boxed{\delta_x=r\big(g-c_1\,\hat x\big),\qquad c_1=\tfrac1d\textstyle\sum_j g_j\hat x_j}
$$

参数梯度（对所有 token 求和，即对 batch 维与序列维归约）：

$$
\delta_w=\sum_{b,t}\delta_y\odot\hat x
$$

**LayerNorm 的对应形式**（用于对比）：

$$
\delta_x=r\big(g-c_2-c_1\,\hat x\big),\quad c_1=\tfrac1d\textstyle\sum_j g_j\hat x_j,\quad c_2=\tfrac1d\textstyle\sum_j g_j,
$$
$$
\delta_w=\sum\delta_y\odot\hat x,\qquad \delta_b=\sum\delta_y
$$

多出的 $-c_2$ 来自中心化，它把梯度中"沿全 1 方向"的分量去掉。

**几何理解**
- RMSNorm 的 $\delta_x$ 是把 $g$ 中沿 $\hat x$ 方向的分量去掉后再乘 $r$。
- 原因是尺度不变性：$\hat x(\alpha x)=\hat x(x)$（$\epsilon\to0$），所以梯度必须与 $x$ 正交，即 $\sum_i x_i\,\delta_{x,i}=0$。这是一个可以数值验证的性质。

**系统视角**
- 反向需要保存 $x$（或 $\hat x$）和每行的 $r$（$N$ 个标量）。
- 反向有两类归约：沿 $d$ 的行内归约（$c_1$），和沿 $N$ 的跨行归约（$\delta_w$）。
- 全是逐元素与归约，没有 matmul，属于访存密集。

#### 4.3 为什么先升到 fp32 再算

| 数据类型 | 风险 |
|---|---|
| fp16 | 范围小（最大约 65504），$|x|>256$ 时 $x^2$ 就**溢出** |
| bf16 | 范围同 fp32，不溢出，但尾数只有 8 位；对 $d$ 项求和时**累加误差大**，$r$ 的精度差 |

做法：`x.to(float32)` 之后做平方、均值、rsqrt，最后 `.to(in_dtype)`（staff 源码就是这样）。这样不管外层是否 autocast，归一化的统计量始终在 fp32 里计算。

#### 4.4 实现

- 先写与定义一一对应的版本（升精度、平方、均值、`rsqrt`、乘 $r$、乘 $w$、转回原类型）。
- 参数名必须是 `weight`（形状 `(d,)`），才能被 `load_state_dict` 加载。
- 再写一个**手写反向**版本（`torch.autograd.Function`），用于验证 4.2 的推导。

#### 4.5 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 前向与 staff 一致 | 同权重同输入，fp32 | `torch.equal` 或误差 < 1e-6 |
| 与 `torch.nn.RMSNorm` 一致 | 构造时显式 `eps=1e-5`（其默认值不同） | `allclose` |
| 手写反向与 autograd 一致 | **float64**，随机输入，比较 $\delta_x$ 和 $\delta_w$ | `allclose` |
| 尺度不变性 | $\text{RMSNorm}(\alpha x)$ 对比 $\text{RMSNorm}(x)$，$\alpha=3$ | 近似相等（$\epsilon$ 影响很小） |
| 梯度与 $x$ 正交 | 计算 $\sum_i x_i\delta_{x,i}$ | 接近 0 |
| LayerNorm 对照 | 手写 LayerNorm 反向，验证 $-c_2$ 项 | 与 autograd 一致 |

#### 4.6 与 Part 1 的关联：混合精度（3.5）

这一章直接对应 3.5 里"哪些算子在 autocast 下的 dtype"的问题。不要凭推测填表，**用 forward hook 打印实测**：

| 对象 | 验证方法 | 要观察的点 |
|---|---|---|
| RMSNorm 的输入 dtype | hook 里打印 `input[0].dtype` | 残差流是 fp32 还是 bf16 |
| RMSNorm 的输出 dtype | hook 里打印 `output.dtype` | 是否等于输入 dtype（代码里 `.to(in_dtype)`） |
| `weight` 的 dtype | `self.weight.dtype` | 参数始终是 fp32 |
| 归一化内部的统计量 | 看源码 | 显式升到 fp32，与 autocast 无关 |

#### 4.7 代价与 kernel 归属

**FLOPs**：每个元素约几次乘加，总量约 $O(N\cdot d)$，与同层 matmul 的 $O(N\cdot d^2)$ 相比小一个 $d$ 的量级，可忽略。

**访存**：这才是重点。staff 的实现是未融合的 PyTorch 小算子，大致是：

| 步骤 | kernel 类型 |
|---|---|
| `x.to(float32)` | 类型转换（输入已是 fp32 时没有） |
| `x.pow(2)` | elementwise |
| `.mean(-1)` | reduction |
| `+ eps`、`rsqrt` | elementwise（很小） |
| `x * rms` | elementwise |
| `self.weight * x` | elementwise |
| `.to(in_dtype)` | 类型转换（同类型时没有） |

约 6 到 8 个 kernel，每个都要完整读写一遍 $N\times d$ 的张量，而融合后的 kernel 只需读一次写一次。

**量级估算**（small，B=4，ctx=512，fp32）：
- 张量大小：$2048\times768\times4$ B ≈ 6.3 MB
- 每个 elementwise kernel 的读写：约 12.6 MB，按 769 GB/s 约 16 µs
- 约 8 个 kernel → 约 0.13 ms；每次前向有 $2\times12+1=25$ 次 RMSNorm → 约 3 ms
- 对比 small 的 forward 预期 matmul 时间约 16 ms，**约增加 20%**，且 kernel 小，发射开销也不可忽略

这是一个可以在 nsys 里验证的预测，也是 3.7（`torch.compile`）的预期收益来源：编译会把这些小算子融合。

**回填第 12 章对照表**：RMSNorm 的 kernel 名、调用次数、占前向时间的百分比。

In [77]:
class RMSNorm(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        eps: float = 1e-5,
        device=None,
        ):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size, device=device))
        self.eps = eps
    
    def forward(self,x):
        # no bias
        in_dtype = x.dtype

        x = x.to(torch.float32)
        rms = torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+self.eps)
        x = x*rms
        return (self.weight*x).to(in_dtype)

### 5. RoPE（旋转位置编码）

**符号**：RoPE 作用在 **Q、K 的每个头**上（拆头之后），维度为 $d_k$（须为偶数）。
基数 $\Theta$ = `rope_theta` = 10000；第 $j$ 对的频率 $\theta_j$；位置 $m,n\in\{0,\dots,T-1\}$。
V 和 embedding 都**不**做 RoPE。

#### 5.1 目标

希望给 $q$（位置 $m$）和 $k$（位置 $n$）各施加一个与位置有关的变换 $f$，使注意力分数只依赖**相对位置**：

$$\langle f(q,m),\,f(k,n)\rangle = g(q,k,m-n)$$

位置信息直接进入 $QK^\top$，不需要额外加到 embedding 上，也没有可学习参数。

#### 5.2 构造与证明

**二维情形**：把向量相邻两个分量 $(x_{2j},x_{2j+1})$ 看作平面上的点，逆时针旋转角度 $m\theta_j$：

$$
\begin{bmatrix}y_{2j}\\y_{2j+1}\end{bmatrix}
=R(m\theta_j)\begin{bmatrix}x_{2j}\\x_{2j+1}\end{bmatrix},\quad
R(\alpha)=\begin{bmatrix}\cos\alpha&-\sin\alpha\\ \sin\alpha&\cos\alpha\end{bmatrix}
$$

整个 $d_k$ 维向量被分成 $d_k/2$ 对，每对用自己的频率 $\theta_j$，等价于一个块对角旋转矩阵。

**复数形式**：令 $z_j=x_{2j}+\mathrm{i}\,x_{2j+1}$，则 $y_j=z_j\,e^{\mathrm{i}m\theta_j}$。

**相对位置性质的证明**（复数版，最简洁）：二维点积等于 $\mathrm{Re}(a\bar b)$，所以

$$
\langle f(q,m),f(k,n)\rangle=\mathrm{Re}\!\big(q e^{\mathrm{i}m\theta}\cdot\overline{k e^{\mathrm{i}n\theta}}\big)
=\mathrm{Re}\!\big(q\bar k\,e^{\mathrm{i}(m-n)\theta}\big)
$$

只依赖 $m-n$。矩阵版同理：$R(m\theta)^\top R(n\theta)=R((n-m)\theta)$，所以
$q^\top R(m\theta)^\top R(n\theta)\,k=q^\top R((n-m)\theta)\,k$。
对 $d_k/2$ 对求和，整体内积仍只依赖 $m-n$。

**两个副产品**：
- 旋转是正交变换，**不改变 $q,k$ 的范数**；
- 位置越近，同一频率下的相位差越小。

#### 5.3 频率设计

$$\theta_j=\Theta^{-2j/d_k},\qquad j=0,\dots,\tfrac{d_k}{2}-1,\qquad \text{波长 }\lambda_j=\tfrac{2\pi}{\theta_j}$$

以 $d_k=64$、$\Theta=10^4$ 为例：

| 通道对 | $\theta_j$ (rad/token) | 波长 (tokens) | 含义 |
|---|---|---|---|
| $j=0$ | 1 | 约 6.3 | 转得最快，分辨相邻位置 |
| $j=15$ | 约 $10^{-2}$ | 约 630 | 中等尺度 |
| $j=31$ | 约 $1.3\times10^{-4}$ | 约 47000 | 几乎不转 |

- **为什么是几何级数**：波长从几个 token 到几万 token 在**对数轴上均匀分布**，让不同通道负责不同距离尺度。这个选择继承自 Transformer 原始的正弦位置编码，是经验设计，不是唯一解；RoFormer 论文还从"内积随相对距离衰减"的角度给出了论证（可自行查阅）。
- **一个值得注意的事实**：训练 ctx 只有 128 到 1024 时，最后几对通道的累计转角很小（$j=31$ 在 $m=1024$ 处只转约 0.14 rad），它们近似当作"与位置无关"的通道。
- **$\Theta$ 的作用**：增大 $\Theta$ 会拉长最大波长，是长上下文外推常用的调节旋钮。

#### 5.4 可微：反向就是逆旋转

RoPE 没有参数，$R$ 是正交矩阵，所以：

$$\delta_x = R(m\theta_j)^\top\delta_y = R(-m\theta_j)\,\delta_y$$

即把 $\sin$ 的符号取反，其余形式与前向相同，不需要保存任何激活（cos/sin 表是常量）。

#### 5.5 算法：不要真的构造旋转矩阵

块对角矩阵乘法每个向量需要约 $2d_k^2$ FLOPs，没有必要。逐对用逐元素公式：

$$y_1=\cos\cdot x_1-\sin\cdot x_2,\qquad y_2=\sin\cdot x_1+\cos\cdot x_2$$

每对 4 次乘法、2 次加减，约每个元素 3 FLOPs。

**cos/sin 缓存**
- 形状 `(2, context_length, d_k/2)`：第 0 维是 cos 和 sin，第 1 维是位置，第 2 维是频率对。
- 构造：`freqs = Θ ** (-arange(0, d_k, 2) / d_k)`，再与 `arange(context_length)` 做外积，取 cos、sin 后 stack。
- 在 staff 源码里以 `register_buffer(..., persistent=False)` 注册，**不进入 state_dict**，且全部层共用同一份。
- 缓存是 fp32，若 Q、K 是 bf16，逐元素乘法会发生类型提升（见 5.8）。

**staff 的布局与教科书不同，但等价**

| | 输入如何配对 | 输出布局 |
|---|---|---|
| 教科书 | 相邻 $(2j,2j+1)$ | 同样交错：$[y_0,y_1,y_2,y_3,\dots]$ |
| staff | 相邻 $(2j,2j+1)$（`(half_d xy)`，xy 变化最快） | **前后半**：$[y_0,y_2,y_4,\dots\ |\ y_1,y_3,y_5,\dots]$ |

staff 输出相当于对教科书输出做一个固定置换 $P$（偶数位在前，奇数位在后）。**Q 和 K 经过同一个置换**，而点积对同时置换两个向量的分量是不变的：
$(PQ')\cdot(PK')=Q'\cdot K'$，所以注意力分数与教科书完全一致。V 不做 RoPE，不受影响。

> 前提：RoPE 之后，Q、K 的最后一维只参与 $QK^\top$ 的点积。任何依赖具体通道顺序的后续操作都会破坏这个等价性。

#### 5.6 实现

- 先按 5.2 写**教科书版**（复数或逐对公式，交错输出），再写 **staff 版**（前后半输出）。
- 接口：`forward(x, pos_ids=None)`。`pos_ids` 为 None 时取缓存的前 `seq_len` 个位置；传入时按位置索引，支持任意位置。
- 在注意力里的位置：Q、K 拆头之后、计算分数之前，$d=d_k$。

#### 5.7 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 缓存一致 | 比较 `_freq_cis_cache` | `allclose` |
| 与 staff 一致 | 同输入同位置，fp32 | `allclose`（误差 < 1e-6） |
| 教科书版与 staff 版 | 对教科书输出取置换 `perm=[0,2,4,…,1,3,5,…]` 后与 staff 输出比较 | `allclose`；**不做置换则不相等** |
| 范数不变 | 比较 $\|y\|$ 与 $\|x\|$（每个头每个位置） | 相等 |
| 相对位置性质 | 把**同一个** $q$ 与**同一个** $k$ 复制到所有位置，得分数矩阵 $S=Q'K'^\top$ | $S[m,n]$ 与 $S[m+s,n+s]$ 相等，$S$ 沿对角线为常数 |
| 位置 0 | $m=0$ 时旋转角为 0 | 输出等于输入（置换后） |
| 手写反向 | float64 下，逆旋转与 autograd 梯度比较 | `allclose` |
| 任意位置路径 | 传入 `pos_ids` 与默认路径比较 | 一致 |

#### 5.8 与 Part 1 的关联：混合精度（3.5）

- Q、K 是线性层输出，autocast 下是 bf16；cos/sin 缓存是 fp32，**预期**逐元素乘法会提升为 fp32，因此 RoPE 的输出可能是 fp32，下一步的注意力 matmul 再被 autocast 转回 bf16。
- 这是推测，**用 forward hook 实测**后再填 3.5 的 dtype 表，不要预设答案。

#### 5.9 代价与 kernel 归属

**FLOPs**：Q、K 共 $2\cdot N\cdot d$ 个元素，每元素约 3 FLOPs，每层约 $6Nd$。small、B=4、ctx=512 时每层约 9.4 MFLOP，不到同层 matmul（约 38.7 GFLOP）的万分之三，可忽略。

**访存**：未融合的实现把一次旋转拆成许多小 kernel，每个都要完整读写一遍张量：

| 步骤 | 数量级 |
|---|---|
| 两个乘法 + 一个减法（$y_1$） | 3 个 elementwise |
| 两个乘法 + 一个加法（$y_2$） | 3 个 elementwise |
| `concat` | 1 个拷贝 |
| 上游 `rearrange` 产生的非连续视图 | 可能带来额外拷贝或跨步访问 |

每次调用至少 7 个 kernel，每层调用 2 次（Q 和 K），反向会更多。

**量级估算**（small，B=4，ctx=512，fp32）：单个 kernel 搬运约 9 MB，按 769 GB/s 约 12 µs；7 个 kernel × 24 次调用 ≈ 2 ms，大约占 forward 预期时间（约 16 ms）的一成，另有 kernel 发射开销。这是一个可在 nsys 里验证的预测，`torch.compile`（3.7）会把它们融合。

**回填第 12 章对照表**：RoPE 的 kernel 名、调用次数、占前向时间的比例。

- 注意两种rearrange方法结果

In [78]:
d = 100
x = torch.arange(d)
print(x)
# x1,x2 = rearrange(x,"... (xy half_d)->xy ... half_d",xy=2).unbind(0)
x1,x2 = rearrange(x,"... (half_d xy)->xy ... half_d",xy=2).unbind(0)
print(x1)

result = einx.id("... x_half, ... x_half -> ... (x_half (1 + 1))", x1, x2).contiguous()
#result = torch.concat((x1, x2), dim=-1)
print(result)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
        72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95, 96, 97, 98, 99])
tensor([ 0,  2,  4,  6,  8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34,
        36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70,
        72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
      

In [79]:
class  RotaryEmbedding(nn.Module):
    def __init__(
        self,
        d_k:int,
        context:int,# N
        theta:float = 10000.0,
        ):
        super().__init__()
        self.d_k = d_k
        self.ctx = context
        self.theta = theta
        self.register_buffer(
            "_freq_cis_cache", RotaryEmbedding._init_cache(context_length, self.d_k, theta), persistent=False
        )
        self._freq_cis_cache: Float[Tensor, "2 context_length half_dim"]

    @staticmethod
    def _init_cache(context: int, d_k: int, theta: float) -> Float[Tensor, " 2 context_length half_dim"]:
        assert d_k % 2 == 0

        i = torch.arange(0, d_k, 2) / d_k
        freqs = torch.tensor(theta) ** -i
        m = torch.arange(context_length)

        freqs = einsum(m, freqs, "m, f -> m f")

        cos, sin = torch.cos(freqs), torch.sin(freqs)
        return torch.stack((cos, sin))

    def forward(self,x):
        # no posi in
        x1, x2 = rearrange(x, "... (half_d xy) -> xy ... half_d", xy=2).unbind(0)
        seq_len = x.size(-2) # only need first seq_len
        cos, sin = self._freq_cis_cache[:, :seq_len, :].unbind(0)

        # 2D rotation matrix applied to pairs in x
        x1_rot = cos * x1 - sin * x2
        x2_rot = sin * x1 + cos * x2
        # result = einx.id("... x_half, ... x_half -> ... (x_half (1 + 1))", x1_rot, x2_rot).contiguous()
        result = torch.concat((x1_rot, x2_rot), dim=-1)
        return result


        

### 6. 因果多头自注意力

**符号**：$B$ batch，$T$ 序列长度，$d$ 模型宽度，$h$ 头数，$d_k=d/h$，$N=B\cdot T$。
单头时 $Q,K,V\in\mathbb{R}^{T\times d_k}$；缩放因子 $s=1/\sqrt{d_k}$。

#### 6.1 单头缩放点积注意力

$$
S=s\,QK^\top,\qquad P=\mathrm{softmax}(S+M),\qquad O=PV
$$

- $S\in\mathbb{R}^{T\times T}$：第 $(i,j)$ 项是"位置 $i$ 的 query 对位置 $j$ 的 key"的匹配分数。
- softmax 对**每一行**（沿 key 维）做，所以 $P$ 的每行是一个概率分布。
- $M$ 是因果 mask（见 6.4）。输出 $o_i=\sum_j p_{ij}v_j$：每个位置是 value 的加权平均。

#### 6.2 为什么除以 $\sqrt{d_k}$

设 $q,k$ 的各分量独立、均值 0、方差 1，则

$$
\mathrm{Var}(q\cdot k)=\sum_{a=1}^{d_k}\mathrm{Var}(q_ak_a)=d_k,\qquad
\mathrm{Var}(s\,q\cdot k)=1
$$

- 不缩放时分数的标准差是 $\sqrt{d_k}$（$d_k=64$ 时为 8），softmax 输入的差距很大，输出接近 one-hot。
- softmax 的雅可比是 $\mathrm{diag}(p)-pp^\top$（见 6.3），one-hot 时趋于 0，**梯度消失**。
- 这个假设在本作业的初始化下近似成立：RMSNorm 输出每个分量的均方为 1，$W_q,W_k$ 的元素方差约 $1/d$，所以 $q,k$ 分量方差约 1（截断正态使它略小于 1）。RoPE 保持范数，不改变这一点。
- 可数值检验：初始化后测 $\mathrm{Var}(S)$，缩放后应略小于 1，不缩放则约为 $d_k$。

#### 6.3 softmax：稳定写法与雅可比

**定义**：$p_i=e^{x_i}/\sum_j e^{x_j}$。

**平移不变性**：对任意常数 $c$，$\mathrm{softmax}(x)=\mathrm{softmax}(x-c)$。取 $c=\max_j x_j$：
- 指数的自变量都 $\le0$，**不会上溢**；
- 最大项恰为 $e^0=1$，分母 $\ge1$，**不会除零**。

| 类型 | $e^x$ 上溢的阈值 | 说明 |
|---|---|---|
| fp32 | $x\gtrsim 88.7$ | 平常较少触发 |
| fp16 | $x\gtrsim 11.1$ | 很容易触发 |
| bf16 | 同 fp32 的范围 | 不上溢，但尾数短，求和误差大 |

**雅可比**：

$$
\frac{\partial p_i}{\partial x_j}=p_i(\delta_{ij}-p_j)
\;\Rightarrow\; J=\mathrm{diag}(p)-pp^\top
$$

- $J$ 对称，且 $J\mathbf{1}=0$（对应平移不变性），所以秩至多 $n-1$。
- **不要显式构造 $J$**（$n\times n$）。反向用向量形式：

$$
\boxed{\delta x=p\odot\big(\delta p-\langle\delta p,\,p\rangle\big)}
$$

只需要 $p$ 和一次行内点积，复杂度 $O(n)$。

#### 6.4 因果 mask

**构造**：$M_{ij}=0$（$i\ge j$），$-\infty$（$i<j$）。源码里用布尔矩阵 `causal_mask = (qi >= kj)`（True 表示保留），再用 `torch.where(mask, scores, -inf)`。

**$-\infty$ 的作用**：$e^{-\infty}=0$，被屏蔽位置的概率**恰好为 0**，不是"很小"。反向时 $\delta s=p\odot(\dots)$，这些位置的梯度也恰好为 0。

**两个边界**：
- 每行至少保留对角线，所以不会出现"整行都是 $-\infty$"。若出现，softmax 会算出 $0/0$，得到 NaN。
- 第 0 行只能看自己，$p_{00}=1$。

**系统含义**：因果 mask 是下三角，上三角的一半是无效计算。vanilla 实现仍然算完整的 $T\times T$ 再屏蔽；FlashAttention 会跳过完全被屏蔽的分块（Part 2 的 M4）。

#### 6.5 注意力的反向（单头）

已知 $\delta O$，逐步反推：

$$
\delta V=P^\top\delta O,\qquad
\delta P=\delta O\,V^\top,\qquad
\delta S=P\odot\big(\delta P-D\mathbf{1}^\top\big),
$$
$$
\delta Q=s\,\delta S\,K,\qquad
\delta K=s\,\delta S^\top Q
$$

其中 $D_i=\sum_j P_{ij}\,\delta P_{ij}$。**化简**：$\delta P_{ij}=\delta o_i\cdot v_j$，所以

$$
D_i=\delta o_i\cdot\sum_jP_{ij}v_j=\delta o_i\cdot o_i
\;\Rightarrow\; D=\mathrm{rowsum}(\delta O\odot O)
$$

- 这个技巧**不需要 $P$ 的行内归约**，只需要 $O$ 和 $\delta O$，是 FlashAttention 反向的核心（Part 2 会用到）。
- backward 有 4 个 matmul（$\delta V,\delta P,\delta Q,\delta K$），forward 有 2 个，所以 backward 的 matmul FLOPs 是 forward 的 2 倍，与 Step 2 的 1:2 一致。
- $P$ 需要在前向保存，这是**显存的主要来源**（见 6.11）。

#### 6.6 多头

1. 投影：$Q=XW_Q^\top$，$K=XW_K^\top$，$V=XW_V^\top$，各是 $d\to d$。
2. 拆头：把最后一维 $d$ 切成 $h$ 段，每段 $d_k$：`(B,T,d) → (B,h,T,d_k)`。
3. 各头独立做 6.1。
4. 拼接：`(B,h,T,d_k) → (B,T,h·d_k)`。
5. 输出投影

In [80]:
def softmax(x, dim=-1):
    rescaled_input = x - torch.max(x, dim=dim, keepdim=True)[0]
    exponentiated_rescaled_input = torch.exp(rescaled_input)
    return exponentiated_rescaled_input / torch.sum(exponentiated_rescaled_input, dim=dim, keepdim=True)

- scaled_dot_product_attention
> 该函数无需训练参数，直接定义即可

In [143]:
def scaled_dot_product_attention(
    Q: Float[Tensor, " ... queries d_k"],
    K: Float[Tensor, " ... keys    d_k"],
    V: Float[Tensor, " ... keys    d_v"],
    mask: Bool[Tensor, " ... queries keys"] | None = None,
)->Float[Tensor,"... d_v"]:
    d_k = K.shape[-1]
    x = einsum(Q,K,"... queries d_k, ... keys d_k -> ... queries keys")/math.sqrt(d_k)
    if mask is not None:
            x = torch.where(mask, x, float("-inf"))
    x = softmax(x, dim=-1)
    return einsum(x,V,"... queries keys, ... keys d_v-> ... queries d_v")

- CausalMultiHeadSelfAttention(nn.Module):
1. `rearrange(X,"... seq (head_nums d_k)-> ... head_nums seq d_k",head_nums=self.head_nums)`

    >这里重排即自动解耦（attn计算定义时只要求了最后两维[s,d_k]）

2. 注意mask构造方法，插入维度使得可以广播

In [144]:
# mask
sequence_length = 100
batch_dims = torch.zeros(3,1)
iota = torch.arange(sequence_length, device=x.device)
qi = rearrange(iota, "query -> query 1")
kj = rearrange(iota, "key   -> 1   key")
causal_mask = qi >= kj  # (query, key)
print(causal_mask)
print(causal_mask.shape)
causal_mask = causal_mask.__getitem__((None,) * len(batch_dims) + (...,))  # Add appropriate leading dimensions
print(causal_mask)
print(causal_mask.shape)

tensor([[ True, False, False,  ..., False, False, False],
        [ True,  True, False,  ..., False, False, False],
        [ True,  True,  True,  ..., False, False, False],
        ...,
        [ True,  True,  True,  ...,  True, False, False],
        [ True,  True,  True,  ...,  True,  True, False],
        [ True,  True,  True,  ...,  True,  True,  True]], device='cuda:0')
torch.Size([100, 100])
tensor([[[[[ True, False, False,  ..., False, False, False],
           [ True,  True, False,  ..., False, False, False],
           [ True,  True,  True,  ..., False, False, False],
           ...,
           [ True,  True,  True,  ...,  True, False, False],
           [ True,  True,  True,  ...,  True,  True, False],
           [ True,  True,  True,  ...,  True,  True,  True]]]]],
       device='cuda:0')
torch.Size([1, 1, 1, 100, 100])


In [145]:
class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(
            self,
            d_model:int,
            head_nums:int,
            positional_encoder: RotaryEmbedding | None = None,
    ):
        super().__init__()
        assert d_model%head_nums==0
        if positional_encoder==None:
            warnings.warn("No positional encoder provided", stacklevel=2)
        self.d_k = d_model//head_nums
        self.d_model = d_model
        self.head_nums = head_nums
        self.d_v = self.d_k

        self.proj_q = Linear(self.d_model,self.head_nums*self.d_k)
        self.proj_k = Linear(self.d_model,self.head_nums*self.d_k)
        self.proj_v = Linear(self.d_model,self.head_nums*self.d_v)

        self.proj_out = Linear(self.head_nums*self.d_v,self.d_model)
        self.positional_encoder = positional_encoder

    def forward(
        self,
        x:Float[Tensor,"... d_model"]
    )->Float[Tensor,"... d_model"]:
        *batch_dims, sequence_length, d_model = x.size()
        assert d_model == self.d_model

        # proj
        Q = self.proj_q(x)
        K = self.proj_k(x)
        V = self.proj_v(x)

        # split
        Q,K,V = (
            rearrange(X,"... seq (head_nums d_k)-> ... head_nums seq d_k",head_nums=self.head_nums) 
            for X in(Q,K,V)
            )# [b,s,h*d_k] -> [b,h,s,d_k]

        # RoPE
        if self.positional_encoder is not None:
            Q = self.positional_encoder(Q)
            K = self.positional_encoder(K)

        # mask
        iota = torch.arange(sequence_length, device=x.device)
        qi = rearrange(iota, "query -> query 1")
        kj = rearrange(iota, "key   -> 1   key")
        causal_mask = qi >= kj  # (query, key)
        causal_mask = causal_mask.__getitem__((None,) * len(batch_dims) + (...,))  # Add appropriate leading dimensions

        # attn
        attn = scaled_dot_product_attention(Q,K,V,causal_mask) #[b,h,s,d_v]

        # concat&proj
        y = rearrange(attn,"... h s d_v -> ... s (h d_v)") #[b,h,s,d_v]->[b,s,h*d_v]
        y = self.proj_out(y) #[b,s,s*d_v]->[b,s,d_model]
        return y


### 7. SwiGLU 前馈网络

**符号**：$N=B\cdot T$ 为 token 总数，$d$ 为模型宽度，$d_{ff}$ 为内层宽度。逐 token 独立计算，没有 token 间的交互。
**约定**：沿用 staff 的行向量写法，`Linear` 是 $y=xW^\top$，权重形状为 `(d_out, d_in)`。

#### 7.1 结构

$$
a=xW_1^\top,\quad b=xW_3^\top,\quad u=\mathrm{SiLU}(a),\quad g=u\odot b,\quad y=gW_2^\top
$$

| 矩阵 | staff 参数名 | 形状 | 角色 |
|---|---|---|---|
| $W_1$ | `w1.weight` | $(d_{ff},d)$ | **门**分支，输出经过 SiLU |
| $W_3$ | `w3.weight` | $(d_{ff},d)$ | **值**分支，不经过非线性 |
| $W_2$ | `w2.weight` | $(d,d_{ff})$ | 降维，把 $d_{ff}$ 投回 $d$ |

注意编号不是按顺序的：门是 $W_1$，值是 $W_3$。源码 `w2(silu(w1(x)) * w3(x))` 里的 `silu` 只作用在 `w1` 上，写反会导致 `load_state_dict` 通过但结果不一致。

#### 7.2 SiLU

$$
\mathrm{SiLU}(x)=x\,\sigma(x),\qquad \sigma(x)=\frac{1}{1+e^{-x}}
$$

**导数**：利用 $\sigma'=\sigma(1-\sigma)$，

$$
\mathrm{SiLU}'(x)=\sigma(x)+x\,\sigma(x)\big(1-\sigma(x)\big)=\sigma(x)\big(1+x(1-\sigma(x))\big)
$$

**性质**（均可数值验证）：
- $\mathrm{SiLU}(0)=0$，$\mathrm{SiLU}'(0)=0.5$；
- $x\to+\infty$ 时趋近 $x$；$x\to-\infty$ 时趋近 0；
- **非单调**：在 $x\approx-1.28$ 处有最小值约 $-0.28$，所以允许小幅负输出；
- 导数最大值略大于 1（约 1.1，在 $x\approx2.4$ 附近）；
- 光滑，处处可导，没有 ReLU 在 0 处的拐点。它也叫 Swish-1（$x\sigma(\beta x)$ 取 $\beta=1$）。

#### 7.3 门控机制

**GLU 的想法**：用一个分支控制另一个分支的通过量。

$$
\mathrm{GLU}(x)=\sigma(xW)\odot(xV)
$$

把门里的 $\sigma$ 换成别的函数，得到一族变体（ReGLU、GEGLU、SwiGLU 等，见 Shazeer 2020《GLU Variants Improve Transformer》）。SwiGLU 取 SiLU 作门，再加一个降维矩阵 $W_2$。

**直觉**：
- 标准 MLP 是 $W_2\,\phi(W_1x)$，非线性只对每个通道**单独**作用；
- 门控是两个**线性投影的逐元素乘积**，引入了乘性交互：同一个 token 在每个通道上被"调制"了多少，由 $W_1$ 决定，内容由 $W_3$ 决定；
- 经验上，在相同参数量或相同计算量下，门控版本的效果一般更好，所以被 LLaMA 一类模型采用。这是实验结论，不是推导出来的。

#### 7.4 反向传播

给定 $\delta y$，按计算图倒推：

$$
\delta g=\delta y\,W_2,\qquad \delta W_2=\delta y^\top g
$$
$$
\delta b=\delta g\odot u,\qquad \delta u=\delta g\odot b,\qquad \delta a=\delta u\odot \mathrm{SiLU}'(a)
$$
$$
\delta x=\delta a\,W_1+\delta b\,W_3,\qquad \delta W_1=\delta a^\top x,\qquad \delta W_3=\delta b^\top x
$$

（$\delta W$ 的矩阵乘法在 $N$ 个 token 上求和，$\delta y$ 等按 `(N, ·)` 展平。）

要点：
- 输入 $x$ **同时流入两个分支**，所以 $\delta x$ 是两条路径的**梯度之和**。
- backward 有 **6 个 matmul**（每个线性层 2 个），forward 有 3 个，所以 backward 的 matmul FLOPs 是 forward 的 2 倍，与 Step 2 一致。
- 反向需要保存：$x$、$a$、$b$、$u$、$g$（以及手写 SiLU 里的 $\sigma(a)$），这就是 7.9 里激活显存的来源。

#### 7.5 参数量与 $d_{ff}$ 的选择

每层 FFN 参数：$3\,d\,d_{ff}$（staff 无 bias）。

| FFN 类型 | 矩阵数 | $d_{ff}$ | FFN 参数 | 整层参数（含注意力 $4d^2$） |
|---|---|---|---|---|
| GPT-2 的 GELU MLP | 2 | $4d$ | $8d^2$ | $12d^2$ |
| SwiGLU，**本作业** | 3 | $4d$ | $12d^2$ | $16d^2$（GPT-2 的 1.33 倍） |
| SwiGLU，LLaMA 惯例 | 3 | $\frac83d$（取 64 的倍数） | $8d^2$ | $12d^2$（与 GPT-2 持平） |

- **本作业的 $d_{ff}$ 取值**：small 3072、medium 4096、large 5120、xl 6400、2.7B 10240，全是 $4d$。
- **这就是 Step 2 里"实测参数量大于 GPT-2 名义规模"的原因**：medium 约 0.42B（名义 0.35B），large 约 0.97B（名义 0.76B）。
- $d_{ff}$ 取 64 的倍数是为了对齐 tensor core 的分块，对 matmul 效率有影响。
- 代码里 `d_ff` 是**独立的超参数**，没有强制等于 $4d$。

#### 7.6 实现

- 先手写 `silu`（`x * sigmoid(x)`），再对照 `F.silu`。
- 三个 `Linear`，参数名 `w1`、`w2`、`w3`，初始化沿用第 3 章的 Xavier 型截断正态（$\sigma=\sqrt{2/(d_{in}+d_{out})}$）。
- `forward`：`w2(silu(w1(x)) * w3(x))`。
- **两种 `silu` 的系统差异**（后面做对比实验，见 7.9）：

| | 手写 `x*sigmoid(x)` | `F.silu` |
|---|---|---|
| kernel 数（forward） | 2（sigmoid、mul） | 1 |
| 访存（单位：一个 $N\times d_{ff}$ 张量的读写次数） | 5 | 2 |
| 保存给反向的张量 | $a$、$\sigma(a)$ | $a$（反向时重算 $\sigma$） |

#### 7.7 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| SiLU 前向 | 与 `F.silu` 比较 | `allclose` |
| SiLU 导数 | 7.2 的公式与 autograd 比较（float64） | `allclose` |
| SiLU 的性质 | 在网格上找最小值 | 位置约 $-1.28$，值约 $-0.28$ |
| 参数量 | `sum(p.numel())` | 等于 $3\,d\,d_{ff}$ |
| 与 staff 一致 | 同权重同输入，fp32 | `allclose`（误差 < 1e-5） |
| 门/值是否写反 | 交换 `w1`、`w3` 的权重，输出应**明显改变** | 与原输出不同，说明二者角色不对称 |
| 手写反向 | 按 7.4 在 float64 下算 $\delta x,\delta W_{1,2,3}$ | 与 autograd `allclose` |
| 梯度检验 | `torch.autograd.gradcheck`（小尺寸，float64） | 通过 |
| 逐 token 独立 | 打乱序列中 token 顺序，输出应同样被打乱 | 成立（FFN 没有 token 间交互） |
| 梯度累加 | 检查 $\delta x$ 是两条分支贡献之和 | 与分别反传再相加一致 |

#### 7.8 与 Part 1 的关联

- **3.5 混合精度**：`Linear`（einsum 底层是批量矩阵乘）在 autocast 下**预期**会变成 bf16；`silu` 与逐元素乘不在 autocast 的类型转换名单里，**预期**沿用输入的 dtype，即 bf16。这是预期，用 forward hook 实测后再填表。
- **3.7 `torch.compile`**：SwiGLU 中间的"sigmoid、mul、mul"是一串逐元素运算，是编译融合的直接对象。
- **一个可做的小实验**：把手写 `silu` 换成 `F.silu`，对比 forward 时间与峰值显存。预期显存少一份 $N\times d_{ff}$，时间略降。

#### 7.9 代价与 kernel 归属

**(1) FLOPs（每层 forward）**

$$
\underbrace{2N\,d\,d_{ff}}_{W_1}+\underbrace{2N\,d\,d_{ff}}_{W_3}+\underbrace{2N\,d\,d_{ff}}_{W_2}=6N\,d\,d_{ff}=24N\,d^2\ \ (d_{ff}=4d)
$$

与注意力合起来，每层 forward 为 $32Nd^2+4NTd$（线性层 $8Nd^2$ 的注意力投影加上注意力 matmul），所以 FFN 占比：

$$
\text{FFN 占比}=\frac{24Nd^2}{32Nd^2+4NTd}=\frac{6d}{8d+T}
$$

| 模型 | $T=128$ | $T=512$ | $T=1024$ |
|---|---|---|---|
| small（$d=768$） | 73.5% | 69.2% | 64.3% |
| large（$d=1280$） | 74.1% | 71.4% | 68.2% |

结论：FFN 是每层 FLOPs 的约三分之二到四分之三，但**不是全部**，注意力的投影和 matmul 合计仍占四分之一以上；T 越大，注意力的占比越高。

**示例**（small，$B=4,T=512$）：每层 FFN 约 29.0 GFLOP，按 $P_{fp32}=33.4$ TFLOPs/s 约 0.87 ms，12 层约 10.4 ms。

**(2) 访存：SwiGLU 中间的逐元素运算**

forward 的三个 elementwise kernel（手写 SiLU）：

| kernel | 读 | 写 | 读写张量数 |
|---|---|---|---|
| `sigmoid(a)` | $a$ | $\sigma$ | 2 |
| `a * sigmoid` | $a,\sigma$ | $u$ | 3 |
| `u * b` | $u,b$ | $g$ | 3 |

合计 8 个 $N\times d_{ff}$ 张量的读写。small 下一个这样的张量是 $2048\times3072\times4\,\text{B}\approx25.2$ MB：
- 每层约 $8\times25.2=201$ MB，按 769 GB/s 约 0.26 ms；12 层约 3.1 ms；
- 对比同层 FFN 的 matmul（约 0.87 ms），**逐元素部分约为其 30%**；
- 融合成一个 kernel（读 $a,b$，写 $g$）只需 3 个张量，节省约 60%（`F.silu` 版本是 5 个，节省约 38%）。

这是可在 nsys 里验证的预测。

**(3) 激活显存：每层保存 5 份 $N\times d_{ff}$**

autograd 为反向保存的张量（手写 SiLU）：

| 张量 | 谁保存 |
|---|---|
| $a=xW_1^\top$ | `x * sigmoid(x)` 的乘法 |
| $\sigma(a)$ | 同上，且 `sigmoid` 自己的反向也要它 |
| $u$ | `u * b` 的乘法 |
| $b$ | 同上 |
| $g$ | $W_2$ 的权重梯度 |

此外还有输入 $x$（$N\times d$，两个分支共用）。所以每层约 $5\,N\,d_{ff}\times$ 字节数，用 `F.silu` 则是 4 份。

fp32，$B=4,T=512$（$N=2048$）：

| 模型 | $d_{ff}$ | 每层 (MB) | 总计 (GiB) |
|---|---|---|---|
| small | 3072 | 126 | 1.4 |
| medium | 4096 | 168 | 3.8 |
| large | 5120 | 210 | 7.0 |

这些只是 FFN 的部分，且是预期值，实际要用 memory snapshot 确认（einsum 内部是否产生额外拷贝会影响份数）。

**(4) 与注意力的激活对比**

注意力分数矩阵每层保存 $c$ 份 $B\,h\,T^2$（$c$ 约 1 到 3，取决于 `softmax` 的实现），FFN 保存 $5Nd_{ff}=20Nd$ 个元素，二者之比：

$$
\frac{\text{注意力分数}}{\text{FFN}}=\frac{c\,B\,h\,T^2}{20\,B\,T\,d}=\frac{c\,T}{20\,d_k}
$$

- $d_k=64$ 时为 $cT/1280$，**与模型宽度无关**；
- $T$ 较小时 FFN 的激活更大，$T$ 超过约 $1280/c$ 之后注意力的 $T^2$ 项接管。

**(5) 对 3.2 的提示：OOM 边界要靠实测确定**

之前只算了静态显存。加上激活（只按 FFN 与注意力分数粗估，**还没算** RMSNorm、RoPE、Q/K/V 等 $N\times d$ 的张量和 logits）：

| 配置 | 静态 (GiB) | FFN 激活 | 注意力分数（$c=2$） | 小计 |
|---|---|---|---|---|
| medium，$T=512$ | 6.3 | 3.8 | 3.0 | 约 13.1，加上其他已贴近 16GB |
| medium，$T=1024$ | 6.3 | 7.5 | 12.0 | 大概率 OOM |
| large，$T=128$（不含 optimizer） | 7.2（权重+梯度） | 1.8 | 0.4 | 约 9.4，可行 |
| large，$T=256$（不含 optimizer） | 7.2 | 3.5 | 1.4 | 约 12.1，加上其他贴近上限 |

做法：
- medium 的完整训练步，**从 ctx=128、256 开始逐步加大**，找到 OOM 的边界。这个边界本身就是 3.2 的状态列。
- large 的 forward+backward 从 ctx=128 开始；完整训练步（含 Adam）在本机必然 OOM。
- 在 notebook 里写一个函数 `est_act_bytes(model, B, T, silu_impl, c)`，输出预估显存，再与 `max_memory_allocated` 对比，误差本身值得分析。

**(6) 回填第 12 章对照表**

| 代码位置 | 预期 kernel | 实测 kernel 名 | 调用次数/层 | 占 forward 比例 |
|---|---|---|---|---|
| `w1(x)`、`w3(x)` | GEMM × 2 | | | |
| `sigmoid` | elementwise | | | |
| `x * sigmoid(x)` | elementwise | | | |
| `silu(...) * w3(x)` | elementwise | | | |
| `w2(...)` | GEMM | | | |

In [146]:
def silu(x: torch.Tensor):
    return x * torch.sigmoid(x)

In [156]:
class SwiGLU(nn.Module):
    def __init__(
            self,
            d_model:int,
            d_ff:int,
            bias:float=None
    ):
        super().__init__()
        self.w1 = Linear(d_model,d_ff)
        self.w3 = Linear(d_model,d_ff)
        self.w2 = Linear(d_ff, d_model)

    def forward(self,x:Float[Tensor,"... d_model"])->Float[Tensor,"... d_model"]:
        a = self.w1(x) #dff
        u = silu(a)
        b = self.w3(x) #dff
        return self.w2(u*b) # dff


### 8. Transformer Block（pre-norm 残差块）

**符号**：$x\in\mathbb{R}^{B\times T\times d}$，$N=B\cdot T$。$A(\cdot)$ 为因果多头自注意力（第 6 章），$M(\cdot)$ 为 SwiGLU（第 7 章），$n_1,n_2$ 为两个 RMSNorm（第 4 章，参数名 `ln1`、`ln2`）。

#### 8.1 结构（staff 的 `TransformerBlock`）

$$
h=x+A\big(n_1(x)\big),\qquad y=h+M\big(n_2(h)\big)
$$

对应源码：

```python
x_attn = self.attn(self.ln1(x));            h = x + x_attn
x_ffn  = self.ffn(self.ln2(h));             y = h + x_ffn
```

- 输入输出形状都是 $(B,T,d)$，block 之间可以任意堆叠。
- 子模块与参数名：`attn.{q_proj,k_proj,v_proj,output_proj}.weight`、`ffn.{w1,w2,w3}.weight`、`ln1.weight`、`ln2.weight`。
- `attn` 在 block 里被调用时**没有传 `token_positions`**，RoPE 用默认的前 $T$ 个位置。
- 整个模型里 RoPE 的 cos/sin 缓存只有**一份**，被所有层的 `attn` 共用（构造时把同一个 `positional_encoder` 传给每个 block）。你的 mini 版本也应该共用，不要每层各建一份。

#### 8.2 pre-norm 与 post-norm

| | post-norm（原论文） | pre-norm（staff，GPT-2/LLaMA 采用） |
|---|---|---|
| 公式 | $y=\mathrm{LN}\big(x+F(x)\big)$ | $y=x+F\big(\mathrm{LN}(x)\big)$ |
| 归一化位置 | 在残差**相加之后**，在主干上 | 在子层**输入前**，在支路上 |
| 主干上有无归一化 | 每层都有 | **没有**，残差流一路保持"未归一化" |
| 梯度路径 | 主路要穿过 $L$ 个 LayerNorm | 主路是纯恒等映射 |
| 训练稳定性 | 深层时需要 warmup 和小心的学习率 | 更稳定，对学习率更宽容 |
| 常见代价 | 无 | 残差流的范数随深度增长；输出前**必须再加一个 `ln_final`** |

**为什么 pre-norm 需要 `ln_final`**：主干上没有归一化，最后一个 block 的输出尺度不受控，直接送进 `lm_head` 会让 logits 的尺度依赖于深度。staff 在 `BasicsTransformerLM` 里 `x = self.ln_final(x)` 后才接 `lm_head`。

> 说明：pre-norm 更稳定，是 Xiong 等人（2020）的分析与大量实践共同支持的经验结论。"pre-norm 的最终效果略逊于调得好的 post-norm" 也有一些报道，不同设置下结论不一致，写 notebook 时标注为经验观察即可。

#### 8.3 残差连接与梯度

**(1) 单个 block 的雅可比**（pre-norm）：

$$
\frac{\partial y}{\partial x}=I+\frac{\partial F}{\partial x}
$$

恒等项 $I$ 保证：无论 $F$ 的梯度多小（甚至为 0），**上游梯度总能原样传到下层**。

**(2) 展开整个网络**：

$$
x_L=x_0+\sum_{l=1}^{L}F_l\big(\mathrm{LN}(x_{l-1})\big)
\;\Rightarrow\;
\frac{\partial x_L}{\partial x_0}=I+\sum_l\frac{\partial F_l}{\partial x_0}
$$

- 把 $x_0$ 到 $x_L$ 的信号路径分成：一条**不经过任何子层的直通路径**，加上若干条经过部分子层的路径。
- 对比没有残差的 $L$ 层网络：梯度是 $L$ 个雅可比的**连乘**，容易指数消失或爆炸；有残差时是**求和**，量级更可控。
- 这是"梯度高速公路"的含义，也是深层网络可训练的主要原因。

**(3) 本 block 的反向公式**（给定 $\delta y$，逐步反推）：

$$
\delta h=\delta y+J_{n_2}^\top J_M^\top\,\delta y,\qquad
\delta x=\delta h+J_{n_1}^\top J_A^\top\,\delta h
$$

其中 $J_M^\top\delta y$ 就是第 7 章的反向，$J_{n}^\top$ 就是第 4 章的反向。要点：
- 残差相加的反向是**分叉**：同一个 $\delta y$ 同时送到两条路径，不需要额外计算。
- 两条路径的梯度回到同一个节点时要**求和**。autograd 里这是一次逐元素加法，每个残差一个 kernel，读两个、写一个 $N\times d$ 张量。
- 两个残差，所以每个 block 的反向有 2 个这样的梯度累加 kernel。

**(4) pre-norm 的副作用：残差流方差随深度增长**

假设每个子层输出的方差为 $\sigma_F^2$，且彼此近似独立，则

$$
\mathrm{Var}(x_l)\approx \mathrm{Var}(x_0)+l\,\sigma_F^2
$$

即**随深度线性增长**。后果：
- 越深的层，单个子层相对于残差流的"贡献比例"越小；
- 这也是为什么输出前要有 `ln_final`，而且每个子层看到的输入（经过 LN）尺度保持一致，不受主干尺度影响。

可以在 notebook 里用实测验证（见 8.7）。

#### 8.4 数据流图与算子分类

**图例**：`[C]` 计算密集（GEMM，高于拐点）；`[C↓]` GEMM 但算术强度偏低（实际访存受限）；`[M]` 访存密集（逐元素、归约、拷贝）。

```
x ──────────────────────────────────────────────┐   (残差流，fp32)
│                                               │
└─► ln1 (RMSNorm)                         [M]   │
      ├─► q_proj                          [C]   │
      ├─► k_proj                          [C]   │
      └─► v_proj                          [C]   │
            │                                   │
          拆头 (B,h,T,d_k)             [M, 可能触发拷贝]
          RoPE(Q), RoPE(K)                [M]   │
          QKᵀ → (B,h,T,T)                [C↓]   │
          ÷√d_k                           [M]   │
          mask (where)                    [M]   │
          softmax                         [M]   │
          P·V → (B,h,T,d_k)              [C↓]   │
          拼头 + contiguous               [M]   │
          output_proj                     [C]   │
            │                                   │
            ▼                                   │
          (+)  ◄────────────────────────────────┘  残差加法               [M]
            │
            ├────────────────────────────────────┐
            └─► ln2 (RMSNorm)             [M]    │
                  ├─► w1 (N×d → N×d_ff)   [C]    │
                  └─► w3 (N×d → N×d_ff)   [C]    │
                        sigmoid           [M]    │
                        x·sigmoid(x)      [M]    │
                        ⊙ w3(x)           [M]    │
                        w2 (N×d_ff → N×d) [C]    │
                          │                      │
                          ▼                      │
                        (+)  ◄───────────────────┘  残差加法             [M]
                          │
                          ▼
                          y
```

**用算术强度判定**（fp32，本机拐点 $P_{fp32}/BW=33.4\text{T}/769\text{G}\approx43$ FLOPs/B；bf16 拐点 $\approx125$）。

强度 $=$ FLOPs ÷ 必须搬运的字节数（读两个输入、写一个输出）。以 small、$B=4$、$T=512$（$N=2048$）为例：

| 算子 | FLOPs | 字节数（fp32） | 强度 (FLOPs/B) | 判定 |
|---|---|---|---|---|
| `q_proj`（$N\times768\to768$） | 2.42 G | 14.9 MB | 约 162 | 计算密集，fp32/bf16 均高于拐点 |
| `w1`（$N\times768\to3072$） | 9.66 G | 40.9 MB | 约 236 | 计算密集 |
| $QK^\top$（单头，$T=512,d_k=64$） | 33.6 M | 1.31 MB | 约 26 | **fp32 低于拐点 43，访存受限** |
| $PV$（单头） | 33.6 M | 1.31 MB | 约 26 | 同上 |
| RMSNorm、RoPE、softmax、SiLU、残差加 | 每元素几 FLOPs | 每元素 8 到 12 B | 小于 1 | 访存密集 |

结论：
- **block 里的 6 个线性层 GEMM（$q,k,v,o,w_1,w_3,w_2$ 中的 7 个）是真正的计算密集型**。
- **注意力的两个 matmul 不是**。如果 $d_k=64$ 且把 $T^2$ 的输出写回显存，强度只有约 $d_k/2$ 到 $d_k$ 的量级，在 bf16 下（拐点 125）更明显。所以 FlashAttention 的收益主要来自**减少读写**，而不是减少 FLOPs。
- "强度"在这里按"每个算子单独把输入输出放在显存里"计算，是最坏情况；小 batch 或融合后会变。

#### 8.5 实现

- 先写 `TransformerBlock(d_model, num_heads, d_ff, positional_encoder)`，直接把第 4、6、7 章写好的组件组合。
- 接口与 staff 保持一致，参数名严格对应 8.1。
- `forward` 里两行加法分开写成 `h = x + ...`、`y = h + ...`，不要写成 in-place（`x += ...`），否则 autograd 保存的张量会被改写，反向报错或结果出错。

#### 8.6 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 与 staff 一致 | 同权重同输入，fp32，比较输出 | `allclose`（误差 1e-5 量级） |
| 与 staff 的梯度一致 | 同一个 loss，比较所有参数梯度与 $\delta x$ | `allclose` |
| **子层归零后是恒等映射** | 把 `attn.output_proj.weight` 与 `ffn.w2.weight` 置 0 | 输出严格等于输入（`torch.equal`） |
| 恒等路径的梯度 | 上一条设置下，$\delta x=\delta y$ | 严格相等 |
| 残差分叉的求和 | 分别只开 attention 支路、只开 FFN 支路，计算 $\delta x$，再与两条都开时比较 | 两条都开时的 $\delta x$ 等于"$\delta y$ + 两条支路各自的贡献" |
| 因果性（block 级） | 修改位置 $t+1$ 之后的输入，比较位置 $\le t$ 的输出 | 一致 |
| 逐 token 的位置依赖 | 打乱输入 token 的位置顺序（不带 RoPE 补偿） | 输出会**改变**，说明 RoPE 起作用 |
| 子层顺序 | 把 `ln1/ln2` 的位置换到 post-norm | 与 staff 输出**不同**，用来确认实现确实是 pre-norm |
| 梯度检验 | `gradcheck`（小尺寸，float64） | 通过 |
| 共用 RoPE 缓存 | 检查多个 block 的 `positional_encoder` 是否同一个对象 | `is` 为 True |

"子层归零后是恒等映射"是 pre-norm 的标志性性质，也是最容易发现主干上误加了 norm 的检验。

#### 8.7 可选实验：残差流与梯度随深度的变化

在 `L=12` 的堆叠上（用 mini 实现，随机初始化，float64 或 fp32），**一次 forward + backward** 记录：

| 量 | 记录方式 | 预期（待实测） |
|---|---|---|
| $\mathrm{Var}(x_l)$ 随层数 | forward hook 取各 block 输出 | pre-norm 下近似线性增长 |
| $\lVert\partial L/\partial x_l\rVert$ 随层数 | backward hook | pre-norm 下大致平缓 |
| 同样的量，改成 post-norm 版本 | 复制一份把 norm 挪到相加之后 | 梯度范数的分布与 pre-norm 不同 |

把三条曲线画在一张图上。这个实验不属于作业，但是对"为什么 pre-norm 更稳"最直接的观察。预期是我基于理论给出的，若实测不符，也值得写进注解里分析原因（例如初始化和截断正态的影响）。

#### 8.8 与 Part 1 的关联

**3.5 混合精度：残差流的 dtype**

预期（**用 forward hook 实测，不要预设**）：

| 对象 | 预期 dtype | 原因 |
|---|---|---|
| embedding 输出，即残差流 $x$ | fp32 | 参数是 fp32，查表不在 autocast 范围内 |
| RMSNorm 输出 | fp32 | `.to(in_dtype)` 恢复输入的 dtype，输入是 fp32 |
| GEMM 的输入 | 被 autocast 转成 bf16 | 线性层在 autocast 的低精度名单里 |
| 子层输出 `x_attn`、`x_ffn` | bf16 | GEMM 的输出 |
| 残差相加 $x+x_{attn}$ | **提升为 fp32** | 类型提升规则：fp32 + bf16 → fp32 |
| `ln_final` 输出、`lm_head` 输入 | fp32 → 被转 bf16 | |
| logits | bf16，除非损失函数里升精度 | |

- 如果预期成立，**残差流在整个网络里一直是 fp32**，bf16 只出现在 GEMM 内部。这也是混合精度能保持训练稳定的原因之一。
- autocast 会在每次 forward 里把 fp32 权重转成 bf16（上下文内有缓存），这会引入额外的 cast kernel 和带宽开销。nsys 里可以看到。

**3.4 nsys 打标**：在 block 里再加两个 NVTX 区间 `attn_sublayer`、`ffn_sublayer`，与第 6 章的三段打标一起，就能把一个 block 的时间分成 {ln1, attn 内部三段, 残差, ln2, FFN} 对照 8.9 的预测。

**3.7 `torch.compile`**：block 内所有 `[M]` 类的小算子是融合的候选，GEMM 不会被融合。预期收益大约等于 8.9 表里"访存密集部分"的占比。

#### 8.9 代价与 kernel 归属

**(1) FLOPs（每个 block，forward）**

$$
\underbrace{8Nd^2}_{q,k,v,o}+\underbrace{24Nd^2}_{w_1,w_2,w_3}+\underbrace{4NTd}_{QK^\top,PV}=32Nd^2+4NTd
$$

小模型（$B=4,T=512$）：

| 部分 | FLOPs (GFLOP) | 理想时间 @33.4 TFLOPs/s (ms) |
|---|---|---|
| $q,k,v,o$ 投影 | 9.66 | 0.29 |
| FFN 三个矩阵 | 28.99 | 0.87 |
| 注意力 matmul | 3.22 | 0.10 |
| **每 block 合计** | **41.87** | **1.25** |

12 个 block 共 502 GFLOP，加上 `lm_head` 的 31.5 GFLOP，恰好是 Step 2 里算出的 534 GFLOP。

**(2) kernel 数量（每个 block，forward，fp32，staff 实现）**

| 部分 | kernel 数 | 说明 |
|---|---|---|
| ln1 | 6 | pow、mean、加 eps、rsqrt、乘 rms、乘 weight |
| q/k/v 投影 | 3 或更多 | einsum 底层可能有额外的 reshape/拷贝 |
| 拆头 + RoPE（Q、K 各一次） | 约 14 | 每次 4 乘、1 减、1 加、1 concat |
| 构造 mask | 2 | `arange`、比较 |
| 注意力核心 | 约 9 | 两个 GEMM + 除法 + where + softmax（取决于 `nn_utils.softmax` 的实现，约 5 个） |
| 拼头 `contiguous` | 1 | 拷贝 |
| output_proj | 1 | |
| 残差加法 | 1 | |
| ln2 | 6 | |
| FFN | 6 | 两个 GEMM + sigmoid + 两个乘 + 降维 GEMM |
| 残差加法 | 1 | |
| **合计** | **约 50** | |

**约 50 个 kernel / block，12 层共约 600 个**，这个数字在你的 nsys 里应该是同一个量级（具体取决于 `softmax` 的实现与 einsum 是否插入拷贝）。

**(3) 单个 block 的时间预测（small，$B=4$，$T=512$，fp32，TF32 关）**

按 33.4 TFLOPs/s 与 769 GB/s 估算：

| 部分 | 理想时间 (ms) | 依据 |
|---|---|---|
| GEMM（全部） | 1.25 | 8.9(1) |
| RMSNorm × 2 | 0.18 | 约 69 MB / 次 × 2 |
| RoPE（Q、K） | 0.17 | 第 5 章估算 |
| 注意力里的 $T^2$ 逐元素（除、where、softmax） | 0.92 | 第 6 章估算，约 7 个 100 MB 的 kernel |
| FFN 的逐元素 | 0.26 | 第 7 章估算 |
| 残差加法 × 2、拼头拷贝 | 0.07 | |
| **访存密集部分合计** | **约 1.6** | |
| **每 block 合计** | **约 2.9** | |

**12 个 block 约 34 ms，加上 embedding、`ln_final`、`lm_head`（约 0.9 ms 的 GEMM）和交叉熵（logits 是 $N\times V$，约 82 MB，几个逐元素 kernel）的访存，forward 约 36 ms。**

对比 Step 2 的"只算 matmul"的理想时间 16 ms，预测 $t_{meas}/t_{pred}\approx2.2$，且主要贡献来自：
- 注意力的 $T^2$ 逐元素：约 11 ms（占 31%）；
- 其余逐元素与归约：约 8 ms；
- GEMM 效率低于平台值：约 +2 ms。

这是**可检验的预测**，误差大约 ±50%，来源是 kernel 数和 `softmax` 的实现细节。

**(4) ctx 变化时的行为**

| 条件 | 主要瓶颈 | 预测 |
|---|---|---|
| small，$T=128$ | GPU 工作量太小，**CPU 发射 kernel 的开销**占主导 | 约 600 个 kernel × 10 到 20 µs ≈ 6 到 12 ms，高于 GEMM 的理想时间（约 3.8 ms），$t_{meas}/t_{pred}$ 可能在 2 到 3 |
| small，$T=512$ | GPU 访存 + GEMM | 约 36 ms，比值约 2.2 |
| small，$T=1024$ | $T^2$ 逐元素显著上升（约 4 倍） | 注意力逐元素占比继续上升 |

**$T=128$ 时的现象**：GPU 在等 CPU。这可以在 nsys 时间线里直接看到：kernel 之间有明显空隙，GPU 利用率低。这是 3.2 里"测量方差大不大、为什么"的一个重要回答，也是 CUDA graph 和 `torch.compile` 的"reduce-overhead"模式要解决的问题。

**(5) 激活显存（每个 block，粗估）**

| 部分 | 保存的张量 | small，$B=4,T=512$，fp32 |
|---|---|---|
| FFN | 5 份 $N\times d_{ff}$（第 7 章） | 126 MB |
| 注意力分数 | $c$ 份 $B\,h\,T^2$，$c\approx1$ 到 3（第 6 章） | 50 到 150 MB |
| 其余 $N\times d$ 的张量（RMSNorm 的中间量、Q/K/V、RoPE 的中间量、拼头后的输出等） | 约 15 到 25 份 | 95 到 160 MB |
| **每 block 合计** | | **约 270 到 440 MB** |
| 12 层合计 | | **约 3.2 到 5.3 GB** |

"其余 $N\times d$ 的张量"的份数是粗估，**用下面的办法精确统计，不要靠估计**：

```python
saved = []
def pack(t):
    saved.append((tuple(t.shape), t.dtype, t.numel() * t.element_size()))
    return t
with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
    y = block(x)
print(sum(b for *_, b in saved) / 2**20, "MiB")
```

这个钩子会列出每个 op 为反向保存的每个张量，可以按形状分类，直接对照 6.11、7.9 与本节的估算，误差就是 notebook 里值得分析的内容。

**(6) 回填第 12 章对照表**

| 区间（NVTX） | 预期 kernel 数 | 实测 kernel 数 | 预测时间 (ms) | 实测时间 (ms) | 占 block 比例 |
|---|---|---|---|---|---|
| ln1 | 6 | | 0.09 | | |
| attention 投影（q/k/v） | 3 | | 0.22 | | |
| RoPE | 14 | | 0.17 | | |
| 注意力核心（scores / softmax / PV） | 9 | | 1.0 | | |
|

In [157]:
class TransformerBlock(nn.Module):
    def __init__(
            self,
            d_model:int,
            head_nums:int,
            d_ff:int,
            eps:float = 0.00001,
            device :any = None,
            bias:float = None,
            positional_encoder : RotrayEmbedding|None = None , # block same
            
    ):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.head_nums = head_nums
        self.positional_encoder = positional_encoder
        self.eps = eps
        self.bias = bias

        self.RMSNorm1 = RMSNorm(self.d_model,eps = self.eps,device=device)
        self.CasalMutiheadAttn = CausalMultiHeadSelfAttention(self.d_model,self.head_nums,self.positional_encoder)
        self.RMSNorm2 = RMSNorm(self.d_model,eps = self.eps,device=device)
        self.SwiGLU = SwiGLU(self.d_model,self.d_ff,bias=self.bias)

    def forward(self,x:Float[Tensor,"... d_model"])->Float[Tensor,"... d_model"]:
        ln1 = self.RMSNorm1(x)
        attn = self.CasalMutiheadAttn(ln1)

        y = x+attn
        ln2 = self.RMSNorm2(y)
        return y+self.SwiGLU(ln2)


### 9. 整体拼装：BasicsTransformerLM

**符号**：沿用符号表；$V=10000$，$N=B\cdot T$。

#### 9.1 结构与数据流

```
token ids (B,T) int64
  └─ token_embeddings  → (B,T,d)        查表，无 FLOPs
      └─ layers[0..L-1] → (B,T,d)        L × TransformerBlock（第 8 章）
          └─ ln_final   → (B,T,d)        RMSNorm
              └─ lm_head → (B,T,V)       Linear: d → V，不绑定
                  └─ loss = cross_entropy(logits.view(N,V), targets.view(N))
```

| 步骤 | 张量 | 形状 | dtype（fp32 训练） |
|---|---|---|---|
| 输入 | `x` | $(B,T)$ | int64 |
| embedding 输出（残差流起点） | | $(B,T,d)$ | fp32 |
| 每个 block 之后 | | $(B,T,d)$ | fp32 |
| `ln_final` 之后 | | $(B,T,d)$ | fp32 |
| logits | | $(B,T,V)$ | fp32 |

- pre-norm 的残差流没有被归一化（第 8 章），所以 `ln_final` 必须在 `lm_head` 之前。
- 返回的是**未归一化的 logits**，不含 softmax；损失在模型外计算。
- 没有 dropout、没有位置 embedding 表（位置信息只来自每层 attention 里的 RoPE）。

#### 9.2 子模块清单与 `state_dict` 键

| 组件 | 键名 | 形状 |
|---|---|---|
| embedding | `token_embeddings.weight` | $(V,d)$ |
| 每层 attention | `layers.{i}.attn.{q_proj,k_proj,v_proj,output_proj}.weight` | $(d,d)$ × 4 |
| 每层 FFN | `layers.{i}.ffn.{w1,w3}.weight`、`ffn.w2.weight` | $(d_{ff},d)$ × 2、$(d,d_{ff})$ |
| 每层 norm | `layers.{i}.ln1.weight`、`layers.{i}.ln2.weight` | $(d,)$ × 2 |
| 最终 norm | `ln_final.weight` | $(d,)$ |
| 输出头 | `lm_head.weight` | $(V,d)$ |

- 条目总数 $=9L+3$：每层 9 个（4+3+2），加上 embedding、`ln_final`、`lm_head`。
  small 为 111，medium 为 219，large 为 327，xl 为 435，2.7B 为 291。
- **RoPE 的 cos/sin 缓存不在 `state_dict` 里**（`persistent=False`），也没有参数。
- `token_embeddings.weight` 与 `lm_head.weight` 形状都是 $(V,d)$，这正是它们**可以**绑定的前提。

#### 9.3 初始化汇总

| 参数 | 初始化 | 数值 |
|---|---|---|
| embedding | 截断正态，std=1，截断 $[-3,3]$ | 实际 std ≈ 0.987 |
| 所有 `Linear`（含 `lm_head`） | 截断正态，$\sigma=\sqrt{2/(d_{in}+d_{out})}$，截断 $\pm3\sigma$ | 实际 std ≈ $0.987\sigma$ |
| 所有 RMSNorm 的 `weight` | 全 1 | |

截断正态的实际方差：$1-\dfrac{2a\,\varphi(a)}{2\Phi(a)-1}$，取 $a=3$ 约为 0.973，std 约 0.987。写成一个检验：**逐参数组统计 std，与公式比较**。

**初始 logits 的尺度与初始 loss**（small，$d=768$）：
- `lm_head` 的 $\sigma=\sqrt{2/(768+10000)}\approx0.0136$；
- `ln_final` 的输出每个分量均方约为 1，所以 logit 的 std 约 $0.0136\times\sqrt{768}\approx0.38$；
- 随机目标下，$\text{loss}\approx\ln V+\sigma_{logit}^2/2\approx9.21+0.07=9.28$。

可以直接当作**第一个 sanity check**：随机 token、随机目标、新初始化的模型，初始 loss 应接近 9.3，而不是 0 或 100。

#### 9.4 参数量：精确公式

$$
P=\underbrace{Vd}_{\text{embedding}}
+L\underbrace{\big(4d^2+3d\,d_{ff}+2d\big)}_{\text{一个 block}}
+\underbrace{d}_{\texttt{ln\_final}}
+\underbrace{Vd}_{\texttt{lm\_head}}
$$

取 $d_{ff}=4d$ 时，一个 block 为 $16d^2+2d$。RoPE 与因果 mask 没有参数。

**手算（small，$d=768$，$L=12$，$d_{ff}=3072$）**：

| 项 | 计算 | 结果 |
|---|---|---|
| 每层注意力 | $4\times768^2$ | 2,359,296 |
| 每层 FFN | $3\times768\times3072$ | 7,077,888 |
| 每层 norm | $2\times768$ | 1,536 |
| 每层合计 | | 9,438,720 |
| 12 层 | | 113,264,640 |
| embedding + `lm_head` | $2\times10000\times768$ | 15,360,000 |
| `ln_final` | | 768 |
| **总计** | | **128,625,408** |

**五档模型**：

| 模型 | blocks | embedding + head | `ln_final` | **总参数** | emb+head 占比 | 条目数 |
|---|---|---|---|---|---|---|
| small | 113,264,640 | 15,360,000 | 768 | **128,625,408** | 11.9% | 111 |
| medium | 402,702,336 | 20,480,000 | 1,024 | **423,183,360** | 4.8% | 219 |
| large | 943,810,560 | 25,600,000 | 1,280 | **969,411,840** | 2.6% | 327 |
| xl | 1,966,233,600 | 32,000,000 | 1,600 | **1,998,235,200** | 1.6% | 435 |
| 2.7B | 3,355,607,040 | 51,200,000 | 2,560 | **3,406,809,600** | 1.5% | 291 |

与 Step 2 的差异：Step 2 的公式没算 norm 权重，差别约万分之一（medium 423.1M 对 423.2M）。

**核对方法**（meta 设备，不占显存）：

```python
with torch.device("meta"):
    m = BasicsTransformerLM(vocab_size=10000, context_length=1024, d_model=d,
        num_layers=L, num_heads=h, d_ff=4*d)
assert sum(p.numel() for p in m.parameters()) == P_formula
assert len(m.state_dict()) == 9*L + 3
```

**一个会误导你的细节**：staff 源码里 `get_num_params` 的文档字符串和构造函数末尾的日志都写着 "non-embedding"，但函数体是 `sum(p.numel() for p in self.parameters())`，返回的是**全部**参数。所以日志里的数字包含 embedding 和 `lm_head`，不要把它当作非 embedding 参数量引用。

#### 9.5 `lm_head` 不绑定，以及绑定会改变什么

**事实**：
- staff 源码里绑定那一行（`self.lm_head.weight = self.token_embeddings.weight`）是**被注释掉的**，并保留了论文里的说明；
- 作业官方选择不绑定的理由，我没有看到原文，**不要凭记忆写进注解**，以你的 handout 为准。

**绑定会改变什么**：

| 方面 | 不绑定（staff） | 绑定 | 备注 |
|---|---|---|---|
| 参数量 | $2Vd$ | $Vd$ | 节省 $Vd$ |
| 节省比例 | | small 6.0%、medium 2.4%、large 1.3%、xl 0.8%、2.7B 0.75% | **越小的模型越划算** |
| 静态显存（权重+梯度+Adam，16 B/参数） | | 省 $16Vd$ B：small 123 MB、medium 164 MB、large 205 MB | 量级很小 |
| **FLOPs** | $2NdV$（`lm_head` matmul） | **不变** | matmul 照样要做，绑定只省参数不省计算 |
| 初始化 | 两个矩阵各用各的尺度 | 只能共用一个 | 见下 |
| 梯度 | embedding 梯度来自查表（只有出现的行非零，但张量是稠密的）；`lm_head` 梯度稠密 | 两路梯度**累加到同一个张量** | |
| `state_dict` | 两个独立键 | 两个键指向**同一份存储** | `parameters()` 会**去重**，只计一次 |
| 加载不绑定的权重 | | 第二个键会覆盖第一个，**logits 不会与 staff 对齐** | 对拍时必须用不绑定版本 |

**初始化尺度冲突**（可数值验证）：

| | embedding（std=1） | `lm_head`（std≈0.0136） |
|---|---|---|
| 初始 logits 的 std（$d=768$） | | 约 0.38 |
| 若直接共用 embedding 的 std=1 | 约 $\sqrt{768}\approx27.7$ | |
| 对应初始 loss | | 约 9.3 → **约 100 量级** |

两个矩阵的"合适初始化"相差约 70 倍，绑定后必须选一个折中（例如 GPT-2 对两者都用 0.02），这会改变 embedding 或 `lm_head` 之一的训练动态。这是绑定在**这份代码**里不是"免费午餐"的直接原因，但不能由此推断官方的动机。

**什么时候绑定划算**：词表占比高的小模型（比如 small 的 emb+head 占 11.9%）；大词表（例如 10 万以上）时节省更显著。本作业 $V=10000$，节省有限。

**对 Part 3、4 的影响**：绑定后同一个参数在 DDP 里只通信一次、在优化器分片里只算一次。如果你的实现是"遍历 `parameters()`"，天然去重；如果是"遍历 `state_dict()`"，会重复。本作业不绑定，不会遇到，但写 DDP 时最好有意识。

#### 9.6 实现要点

- **构造顺序**：先 embedding，再按 `d_head=d/h` 创建**一个** `RotaryEmbedding`，把同一个对象传给每一个 block，再建 `ln_final` 和 `lm_head`。
- **RoPE 缓存只有一份**，被所有层共用；大小为 $2\times\text{ctx}\times d_k/2$ 个 fp32，ctx=1024、$d_k=64$ 时只有约 262 KB。
- **`context_length` 是硬约束**：`RotaryEmbedding.forward` 在位置为 None 时取缓存的前 $T$ 行；若 $T>\text{context\_length}$，缓存只有 `context_length` 行，**后面的广播会报形状错误**。
  - benchmark 扫 ctx 时，要么每个 ctx 重建模型，要么一次建成 `context_length=1024`（短序列取前缀即可）。
  - 对 3.2 来说二者数值上等价，缓存大小可以忽略。
- `forward` 里不要在 block 内用 in-place 的残差加法（第 8 章）。
- 不需要实现 `generate`、`from_pretrained`。顺带一提，staff 的 `generate` 里 `masked_fill` 的结果没有赋值，top-k 实际不生效；与本作业无关。
- `rope_theta=None` 会关闭 RoPE，模型将没有任何位置信息，注意力还会发出警告，不要误传。

#### 9.7 对拍

| 检查 | 方法 | 预期 |
|---|---|---|
| 参数量 | meta 设备，与 9.4 的精确公式 | 完全相等 |
| `state_dict` 键集合 | 与 staff 比较 | 完全相同，条目数 $9L+3$ |
| **整体 logits** | `mini.load_state_dict(staff.state_dict(), strict=True)`，同一批 token | `allclose`，fp32 下最大绝对误差 1e-5 量级 |
| 整体梯度 | 同一个 loss，对所有参数逐个比较 | `allclose` |
| 定位失败 | 对每个 block 的输出挂 forward hook，逐层比较，二分 | 找到第一个出现差异的模块 |
| 初始化统计 | 逐参数组统计 std | 与 9.3 的公式在几个百分点内一致 |
| **初始 loss** | 随机 token、随机目标、新初始化 | 约 9.3（$\ln V+\sigma^2/2$） |
| 因果性（整模型） | 改动位置 $t+1$ 之后的 token，比较位置 $\le t$ 的 logits | 一致 |
| 不绑定 | `lm_head.weight is not token_embeddings.weight` | 为 True |
| 绑定实验（可选） | 把绑定打开 | 参数量恰好少 $Vd$；加载 staff 权重后 logits **不一致**；初始 loss 与 9.3 的预测一致 |
| RoPE 共享 | 各 block 的 `attn.positional_encoder is model.positional_encoder` | 为 True |
| `T > context_length` | 故意构造 | 报形状错误（确认约束理解正确） |

#### 9.8 与 Part 1 的关联

- **benchmark 的"forward"区间包含损失计算**（如果你把 `cross_entropy` 放在 forward 的 NVTX 里）。作业问的 forward 时间是否包含 loss，要看你的 handout；建议 NVTX 拆成 `forward`、`loss` 两个区间，两边都能回答。
- **3.5 混合精度**：`lm_head` 的 matmul 在 autocast 下预期是 bf16，所以 **logits 预期为 bf16**。loss 的 dtype 取决于 `nn_utils.cross_entropy` 内部有没有升精度（手写算子不在 autocast 自动升精度的名单里），用 hook 或 `print(dtype)` **实测**，不要预设。
- **模型规模与 `lm_head` 的比重**：$V$ 固定为 10000，所以 `lm_head` 的相对重要性随 $d$ 增大而下降（9.9 的表），不同模型 profile 的形状会明显不同。
- **3.8 显存**：logits 的显存在本作业中**不是**大头（9.9），不要把"logits 占显存"的结论从大词表模型照搬过来。

#### 9.9 代价与 kernel 归属（整模型）

**(1) FLOPs（forward）**

$$
F_{fwd}=L\big(32\,N d^2+4\,N T d\big)+2\,N d V
$$

（$d_{ff}=4d$；embedding 查表无 FLOPs。）训练一步为 $3F_{fwd}$。

**$B=4$，$T=512$ 的汇总**：

| 模型 | blocks (TFLOP) | `lm_head` (GFLOP) | forward 合计 (TFLOP) | `lm_head` 占比 |
|---|---|---|---|---|
| small | 0.502 | 31.5 | 0.534 | 5.9% |
| medium | 1.752 | 41.9 | 1.794 | 2.3% |
| large | 4.059 | 52.4 | 4.111 | 1.3% |
| xl | 8.375 | 65.5 | 8.441 | 0.8% |
| 2.7B | 14.087 | 104.9 | 14.192 | 0.7% |

对应 Step 2 的数值，这里是把它按模块拆开。

**(2) 静态显存（精确参数量，16 B/参数，GiB）**

| 模型 | 参数 (fp32) | 权重+梯度+Adam |
|---|---|---|
| small | 0.48 | 1.92 |
| medium | 1.58 | 6.31 |
| large | 3.61 | 14.45 |
| xl | 7.44 | 29.78 |
| 2.7B | 12.69 | 50.77 |

**(3) logits 与损失的显存**（$B=4$）

| 配置 | 单个 $(N,V)$ fp32 张量 |
|---|---|
| $T=512$ | 82 MB |
| $T=1024$ | 164 MB |

- 保存的 logits、softmax 中间量、反向的 $\delta$logits，合计约 3 到 4 份，约 0.25 到 0.65 GB。
- 相比之下，small 的所有 block 激活约 3 到 5 GB（第 8 章），**logits 只占一成左右**。
- 若 $V$ 放大到 15 万，这一项会放大 15 倍，成为主要显存项，这就是大词表 LLM 常用"分块计算损失"的原因。

**(4) 整模型 forward 时间预测（small，$B=4$，$T=512$，fp32，TF32 关）**

| 部分 | 预测 (ms) | 依据 |
|---|---|---|
| 12 个 block | 34.8 | 第 8 章：约 2.9 ms/block |
| embedding 查表 | 约 0.02 | 读写约 6 MB |
| `ln_final` | 约 0.09 | 同一个 RMSNorm 的估算 |
| `lm_head` GEMM | 0.94 | 31.5 GFLOP ÷ 33.4 TFLOPs/s |
| 交叉熵（logits 82 MB，约 5 趟读写） | 约 1.1 | 取决于 `nn_utils.cross_entropy` 的实现 |
| **forward（含 loss）** | **约 37** | 对 Step 2 的 16 ms 理想值，比值约 2.3 |

这是预测，误差大约 ±50%，主要不确定性是 softmax 与交叉熵的实现。

**(5) kernel 数量（forward，fp32）**

约 $50\times L$（blocks）$+1$（embedding）$+6$（`ln_final`）$+1$（`lm_head`）$+$ 交叉熵若干。small 约 610 到 620 个。

**(6) 回填第 12 章对照表**

| NVTX 区间 | 预期 kernel 数 | 实测 | 预测时间 (ms) | 实测时间 (ms) | 占 forward 比例 |
|---|---|---|---|---|---|
| embedding | 1 | | 0.02 | | |
| 12 × block | 约 600 | | 34.8 | | |
| `ln_final` | 6 | | 0.09 | | |
| `lm_head` | 1 | | 0.94 | | |
| loss | 数个 | | 1.1 | | |
| **合计（forward）** | 约 615 | | **约 37** | | |

In [158]:
class BasicsTransformerLM_(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        context_length: int,
        d_model: int,
        num_layers: int,
        num_heads: int,
        d_ff: int,
        rope_theta: float | None = 10_000.0,
    ):
        assert d_model%num_heads == 0
        super().__init__()
        self.d_k = d_model//num_heads
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.context_length = context_length
        self.rope_thete = rope_theta
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.positional_encoder = RotaryEmbedding(self.d_k,context=self.context_length)

        self.embedding = Embedding(self.vocab_size,self.d_model)
        self.layers = nn.ModuleList(
            TransformerBlock(
                d_model=self.d_model,          
                head_nums=self.num_heads,
                d_ff=self.d_ff,
                positional_encoder=self.positional_encoder
                ) 
            for _ in range(self.num_layers)
        )
        self.RMSNorm = RMSNorm(self.d_model)
        self.output_proj = Linear(self.d_model,self.vocab_size)

    def forward(self,tokens:Int[Tensor,"... seq_len"])->Float[Tensor,"... seq_len vocab"]:
        x = self.embedding(tokens)
        for l in self.layers:
            x = l(x)
        x = self.RMSNorm(x)
        logits = self.output_proj(x)
        return logits


# 10. 校验：让你的实现或加载与 HF 参照对齐

> **适用场景**：要用 HF 上的某个模型，但不直接用 `transformers` 的前向，而是自己手写、移植到推理引擎、替换 kernel、量化、改结构，或把权重转成别的格式。
> **核心原则**：把 HF 官方实现当作真值。同权重、同输入，你的输出必须与它一致；不一致，就逐层二分，找到第一个出错的位置。

---

## 10.1 校验分层

| 层级 | 问题 | 方法 |
|---|---|---|
| L0 静态 | 结构和权重对得上吗 | config 核对、`state_dict` 键与形状、`strict=True` 加载 |
| L1 整模型前向 | logits 一致吗 | 同权重同 token，比 logits |
| L2 逐层定位 | 不一致时错在哪 | hook 或 `output_hidden_states` 逐层二分 |
| L3 数值 | 差异是 bug 还是舍入 | fp64 参照、误差预算、bf16 容差 |
| L4 行为 | 实际用起来对吗 | 贪心生成、KV cache、padding、tokenizer |
| L5 训练（可选） | 微调后仍一致吗 | 轨迹对拍、单 batch 过拟合 |

L0 到 L2 是主线，大多数问题在这里解决。L3 判断"差异"算不算问题，L4 覆盖前向对拍测不到的使用层面问题。

---

## 10.2 环境设置（每个对拍脚本开头）

```python
import torch
torch.backends.cuda.matmul.allow_tf32 = False    # TF32 带来约 1e-3 误差，必须关
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
torch.manual_seed(0)

def rel(a, b):   # 统一度量：相对误差，在 fp64 中算
    a, b = a.double(), b.double()
    return ((a - b).norm() / b.norm().clamp_min(1e-12)).item()
```

- 同一设备对拍。CPU 与 GPU 的累加顺序不同，会混入误差。
- 同时报告 `rel` 和 `(a-b).abs().max()`。
- HF 参照用 `attn_implementation="eager"`，避免 SDPA 或 flash 内核引入额外差异，**加载后确认确实生效**：`print(hf.config._attn_implementation)`。
- **权重 dtype**：很多模型只发布 bf16 权重。`float32` 加载是精确上采样，两边用同一组数值，所以 fp32 对拍的容差**不需要**放宽。只有拿 bf16 前向去比时才用 bf16 容差。

---

## 10.3 主流程

### 步骤 0：只读 config，核对架构（不下载权重）

```python
from transformers import AutoConfig
c = AutoConfig.from_pretrained(name)     # 只下载 config.json
print(c)
```

先逐项确认你的实现覆盖了模型的全部结构特征，漏一项就对不上：

| 检查项 | 常见差异 |
|---|---|
| `model_type` | 决定整体结构，不是 `llama` 就要逐项核对 |
| 注意力 bias | Qwen2 系列 qkv 有 bias |
| QK 归一化 | Qwen3 系列有 QK-norm |
| `rope_scaling` | Llama 3.x、YaRN、linear、dynamic 等都改变频率 |
| `rope_theta`、`partial_rotary_factor` | 部分模型只对一部分维度做 RoPE |
| 注意力头 | MHA、GQA、MQA（`num_key_value_heads`） |
| `rms_norm_eps` | 各模型不同，不要用默认值 |
| norm 形式 | 有的模型是 `(1+weight)` 缩放（如 Gemma） |
| 激活与 MLP | SwiGLU、GeGLU，gate 与 up 的顺序 |
| `tie_word_embeddings` | True 则 `lm_head` 与 embedding 共享 |
| sliding window、logit soft-cap | 部分模型有，需单独实现 |
| qkv 是否融合 | GPT-NeoX、Phi、Falcon 等融合存储，排布方式各不同 |
| MoE | 路由、共享专家、top-k 归一化 |

新版 `transformers` 的字段名和位置会变（如 `rope_theta` 可能挪到子配置），先 `print(c)` 看实际结构。

### 步骤 1：加载参照，打印键与形状

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
hf  = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float32,              # 旧版本用 torch_dtype=；看 warning，两个名字都试
        attn_implementation="eager").cuda().eval()
tok = AutoTokenizer.from_pretrained(name)
for k, v in hf.state_dict().items(): print(k, tuple(v.shape))
```

访问不畅：`export HF_ENDPOINT=https://hf-mirror.com`，或本机下载后 `scp`，再 `from_pretrained(本地目录)`。自定义结构的模型需要 `trust_remote_code=True`，此时参照就是它自带的代码。

### 步骤 2：权重映射（改名、重排、拆合、绑定）

你的模型的 `state_dict` 键由 `__init__` 里的属性名和嵌套结构决定。写一个转换函数，把 HF 权重变成你的格式：

| 类型 | 说明 |
|---|---|
| 改名 | `self_attn.q_proj`、`mlp.gate_proj` 等对应到你的命名 |
| 重排 | 最典型的是 RoPE 配对，见下 |
| 拆合 | 融合 qkv 与分开的 q、k、v 互转；GQA 的 K、V 复制展开成 MHA |
| 绑定 | tied 时把 embedding 复制给 `lm_head`，或让两者共享同一参数 |

绑定的写法（已加载模型的 `state_dict` 通常两个键都有；磁盘文件可能只有一个，所以用 `.get`）：

```python
sd = hf.state_dict()
emb = sd["model.embed_tokens.weight"]
lm_head = sd.get("lm_head.weight", emb)
```

**RoPE 配对是最容易出错的点**：

| | HF Llama 系 | 相邻配对实现 |
|---|---|---|
| 配对 | 第 $j$ 维与第 $j+d_k/2$ 维（前后半，`rotate_half`，GPT-NeoX 风格） | 第 $2j$ 维与第 $2j+1$ 维（Meta 原始代码，GPT-J 风格） |
| 频率 | $\Theta^{-2j/d_k}$ | 相同 |

如果你的实现用相邻配对，就对 `q_proj`、`k_proj` 的输出行在每个头内置换：新行 $2j$ 取旧行 $j$，新行 $2j+1$ 取旧行 $j+d_k/2$。

**为什么只动行**：线性层 $y=Wx$ 的输入 $x$ 是上一层的隐藏状态，其维度含义不变；RoPE 作用在输出 $y$ 上，所以只需重排输出维度，也就是 $W$ 的行。列（输入维度）不能动。V 不做 RoPE，不置换。如果你的实现本身就用前后半配对，则不需要置换，只改名。

```python
perm = lambda w, n: w.view(n, 2, dk // 2, -1).transpose(1, 2).reshape(n * dk, -1)
rep  = lambda w, n: w.view(n, dk, -1).repeat_interleave(h // n, dim=0).reshape(h * dk, -1)
# q: perm(q_w, h)      k: rep(perm(k_w, kv), kv)      v: rep(v_w, kv)
# 先置换、后展开；MHA 时 rep 是恒等。
# 玩具验证：dk=4 时，行序 [0,1,2,3] 经 perm 变为 [0,2,1,3]
```

### 步骤 3：严格加载，对 logits

```python
mine.load_state_dict(convert(hf.state_dict(), c), strict=True)
for m in mine.modules():                           # 同步 eps
    if isinstance(m, MyRMSNorm): m.eps = c.rms_norm_eps

ids = tok(text, return_tensors="pt").input_ids.cuda()   # 两边必须喂同一份 ids
with torch.no_grad():
    lh = hf(ids, output_hidden_states=True); lm = mine(ids)
print("logits rel", rel(lm, lh.logits))
```

| API | 用途 |
|---|---|
| `state_dict()` | 参数名到张量的有序字典 |
| `load_state_dict(sd, strict=True)` | 缺键、多键、形状不符都会报错，本身就是结构检查 |
| `load_state_dict(sd, strict=False)` | 返回 `missing_keys`、`unexpected_keys`，调试用 |
| 自定义层的参数 | 必须是 `nn.Parameter`，否则不进 `state_dict` |
| `register_buffer(..., persistent=False)` | 不进 `state_dict`，两边要一致 |

报错读法：Missing key 说明命名不对或 qkv 被融合，Unexpected key 说明你多了参数（如 bias），形状转置说明 Linear 权重应为 `(out, in)`。

### 步骤 4：不一致时逐层二分

```python
def capture(model, names):
    store, hs = {}, []
    for n, m in model.named_modules():
        if n in names:
            hs.append(m.register_forward_hook(
                lambda m, i, o, n=n: store.__setitem__(n, (o[0] if isinstance(o, tuple) else o).detach())))
    return store, hs
# 跑一次前向，再按 store 的键顺序（即 hook 触发的执行顺序）逐个 rel 比较，
# 找第一个超阈值的位置；用完 for h in hs: h.remove()
```

- `o[0]` 的处理是因为某些模块（如返回 attention 权重的 attention）输出 tuple。
- 被调用多次的模块，store 里只保留最后一次。
- 也可以直接用 `hf(..., output_hidden_states=True)`：`hidden_states[0]` 是 embedding 输出，`hidden_states[i]` 是第 $i$ 层之后。**最后一项在 HF 里已经过最终 norm**，要和你的 final-norm 输出比，不是和最后一层输出比。

找到第一个出错的层后，在层内继续拆（norm、q/k/v、RoPE 后、attention 输出、MLP）。梯度的定位同理：比 `named_parameters()` 的梯度，从最后一层往前找第一个出错的参数。

### 步骤 5：数值容差与误差预算

**典型量级**（经验值，非保证）：

| 比较 | fp64 | fp32（TF32 关） | bf16 |
|---|---|---|---|
| 相同数学的两个实现 | <1e-12 | 1e-7 到 1e-5 | 1e-3 到 1e-2 |
| fp32 对 fp64 参照 | | 1e-6 到 1e-5 | |
| TF32 开启的 fp32 | | 约 1e-3 | |

**误差预算法**（避免盲目设阈值）：

1. 构造 fp64 参照，算参照 logits $r$；
2. 算 $e_h=\mathrm{rel}(\text{hf}_{32},r)$、$e_m=\mathrm{rel}(\text{mine}_{32},r)$、$e_{mh}=\mathrm{rel}(\text{mine}_{32},\text{hf}_{32})$；
3. 判据：$e_{mh}\lesssim\max(e_h,e_m)$，实际可留 2 到 3 倍余量。

若 $e_h,e_m\sim10^{-6}$ 而 $e_{mh}\sim10^{-2}$，一定是 bug，不是舍入。

**fp64 参照的坑**：

- 要在 `torch.set_default_dtype(torch.float64)` 下构造，而不是构造后 `.double()`，否则按默认 dtype 创建的 buffer（RoPE 的 `inv_freq` 或 cos/sin 缓存）仍是 fp32 精度。
- 但即使这样也不一定够：很多实现（包括 HF 的 rotary 前向）会显式 `.float()`。构造后检查所有 buffer：

```python
for n, b in hf64.named_buffers(): print(n, b.dtype)
```

- 如果 HF 的 fp64 参照下限卡在 1e-7 量级，就改用你自己的实现跑 fp64，与 HF 的 fp32 比，只要 $e_{mh}$ 不超过 fp32 自身的噪声即可。

对**训练好的模型**，激活更大、注意力更尖锐，误差比随机权重时大一点是正常的，用相对误差，别照搬随机权重的阈值。bf16 下以 argmax 一致率和 loss 为主要判据，逐元素误差只作参考。

### 步骤 6：行为级检验（前向对拍测不到的部分）

| 检查 | 做法 | 常见问题 |
|---|---|---|
| 贪心生成 | 显式 `do_sample=False, num_beams=1, use_cache=False`，或手写 argmax 循环，生成 20 到 30 个 token，逐 token 比 | `generation_config` 自带 top_p、temperature，默认不是纯贪心，会造成伪 bug；文本不通顺说明有系统性错误 |
| KV cache 一致性 | 带 cache 的增量解码与无 cache 全量前向的 logits 比较 | 位置索引错、cache 布局错 |
| padding 与 mask | batch 内不同长度，比较有效位置的输出 | padding 侧（left 或 right）、mask 广播错 |
| tokenizer | 两边用同一份 `ids`，检查 BOS、EOS、特殊 token | 一边自动加 BOS，另一边没加 |
| chat template | 对话模型用 `apply_chat_template` 构造输入 | 手拼 prompt 格式不对，效果变差但不报错 |
| 自然文本 loss | 在一段真实文本上算平均 loss | 应远低于 $\ln V$，两边一致到 4 位有效数字 |

### 步骤 7（可选）：微调场景的训练级校验

- **轨迹对拍**：同一份初始权重、同一串 batch、同一个优化器与超参，各训约 50 步。前几步 loss 相对差 ≲1e-5，50 步后参数相对差 ≲1e-3。fp64 下任何明显差异都是 bug。若两边逐位相同，结果就是 0.0。
- **单 batch 过拟合**：固定很小的 batch（如 B=4, T=64），训 100 到 300 步，loss 应单调降到接近 0，用来发现梯度流不通（某支路被 `detach`、in-place 操作）。随机 token 的普通训练 loss 停在 $\ln V$ 附近是正常的。

---

## 10.4 症状速查

| 症状 | 最可能的原因 |
|---|---|
| `load_state_dict` 缺键或多键 | 命名不一致；qkv 融合或拆分不对；多了 bias；漏了 tie |
| size mismatch 且形状转置 | Linear 权重应为 `(out, in)` |
| logits 全 NaN | mask 整行 $-\infty$；softmax 没减最大值；`eps` 放在根号外；fp16 溢出 |
| 差约 1e-3 | TF32 没关；某处混入 bf16 或 fp16 |
| 差 $O(1)$，第 0 层就错 | embedding、第一个 norm、QKV 投影、RoPE 配对、`rope_theta`、`rope_scaling` |
| 位置 0 对，其余错 | RoPE 的位置或频率 |
| 前 $t$ 个位置对，之后错 | 因果 mask 方向或位置索引 |
| 差异随层数放大 | `eps`、norm 的 dtype、RoPE 缓存 dtype、缩放顺序 |
| 只有最后的 logits 错 | `tie_word_embeddings`、final norm 的对应关系、logit soft-cap |
| 前向对、生成乱码或变差 | tokenizer 的 BOS、chat template、KV cache、generation config |
| 单条对、batch 错 | padding 侧、attention mask |
| 前向对、梯度错 | in-place 残差；误用 `.detach()`；手写反向有误 |
| 门控互换后输出不变 | SiLU 作用在了错误的分支 |

**因果性检验**：改动位置 $t$ 之后的 token，位置 $\le t$ 的输出必须不变。

---

## 10.5 通过标准与"检验本身"的检验

| 检查 | 标准（fp32，TF32 关） |
|---|---|
| strict 加载 | 通过，参数量一致 |
| logits 相对误差 | ≲1e-4 |
| 逐层隐藏状态 | 误差不随层数明显放大 |
| next-token argmax 一致率 | 100%（允许并列概率下的个别差异） |
| 自然文本平均 loss | 两者一致到 4 位有效数字，且远低于 $\ln V$ |
| 贪心生成 20 到 30 token | 逐 token 相同，文本通顺 |
| KV cache 增量解码 | 与全量前向一致 |

结果恰好为 0.0 或"没有任何报警"时，先确认比较确实发生了（比较的参数个数、映射数量），再做**反例检验**，确认你的检验体系有鉴别力。每条都应该让误差明显变大：

| 反例 | 检验什么 |
|---|---|
| 扰动一个参数 `p.add_(1e-3 * torch.randn_like(p))` | 比较逻辑本身是否起作用 |
| 去掉 q/k 的 RoPE 置换 | RoPE 配对的差异确实存在，且被检验发现（应为 $O(1)$，生成乱码） |
| 互换 SwiGLU 的 gate 与 up | MLP 路径的判别力 |
| 故意用错误的 `rms_norm_eps` | 容差的灵敏度：eps 偏差引起的误差很小，接近 fp32 噪声，测不出来就说明阈值太松或没有同步 eps |

每次做完要恢复权重。

bf16 同理：autocast 下输出 dtype 应为 bf16，且与 fp32 的差约 1e-3 到 1e-2，否则说明 autocast 被你的自定义层绕过了。

---

## 10.6 补充：参照本身可能有错（共有错误）

上面的流程以 HF 实现为真值，只能保证你和它一致。如果你照着它的代码写，两边可能犯同样的错；HF 实现也偶尔有版本相关的 bug，或与官方实现不一致。这种情况需要不依赖该参照的证据：

| 方法 | 做法 |
|---|---|
| 语义检验 | 训好的模型在自然文本上 loss 应远低于 $\ln V$，贪心生成应通顺。结构性错误（如 RoPE 约定错）会让它明显变差 |
| 对照已发布指标 | 在模型卡或论文报告的数据集上算 perplexity 或基准分数，与公布值比较 |
| 第二个独立实现 | 与 vLLM、llama.cpp、官方原始仓库的输出比较；两个独立实现一致，共有错误的可能性很小 |
| 对照论文或官方代码 | 逐项核对 config 字段的含义，如 RoPE 约定、norm 位置、缩放方式 |
| 随机初始化的局限 | 随机权重下激活小、注意力近似均匀，很多问题暴露不出来，所以优先用训练好的权重对拍 |

经验上，**对拍一致但生成质量明显下降**，几乎一定是这类共有问题，或是步骤 6 里的使用层面问题（tokenizer、template）。

---

## 10.7 金标准与回归测试

校验通过后存下来，之后改任何东西（FlashAttention、`torch.compile`、融合 kernel、量化）都复用：

```python
torch.save({"name": name, "ids": ids.cpu(),
            "logits": hf(ids).logits.detach().cpu()},      # fp32；bf16 另存一份
           "results/golden/model_fp32.pt")
```

每个精度各存一份；回归时同权重同输入，按步骤 5 的容差比较。

---

## 10.8 验收清单

- [ ] config 逐项核对，非标准特征（bias、QK-norm、rope_scaling、tie、sliding window 等）都已覆盖
- [ ] 实际生效的 attention 实现已确认（`hf.config._attn_implementation`）
- [ ] `load_state_dict(strict=True)` 通过，参数量一致
- [ ] fp32 下 logits 与参照在容差内一致，满足误差预算；fp64 参照的 buffer dtype 已核对
- [ ] 四类反例检验都能报警；bf16 autocast 确认生效
- [ ] 逐层隐藏状态误差不随层数放大
- [ ] 贪心生成（`do_sample=False`）逐 token 一致，KV cache 与全量前向一致，padding 与 batch 正确
- [ ] tokenizer、BOS、chat template 两边一致
- [ ] 自然文本 loss 远低于 $\ln V$（必要时按 10.6 补充独立验证）
- [ ] （微调场景）轨迹对拍一致，单 batch 过拟合到接近 0
- [ ] 金标准文件已保存

- 固定环境

In [159]:
import torch
torch.backends.cuda.matmul.allow_tf32 = False      # TF32 会带来约 1e-3 的误差，必须关
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
torch.manual_seed(0)

from contextlib import contextmanager
@contextmanager
def default_dtype(dt):
    old = torch.get_default_dtype(); torch.set_default_dtype(dt)
    try: yield
    finally: torch.set_default_dtype(old)

- `load_state_dict(strict=True)` 加载通过

>加载staff模型

In [160]:
import os, sys
print(os.getcwd())        # kernel 当前工作目录
print(sys.executable)     # kernel 使用的 Python 环境（确认没有装错环境）


/data2/c220/hz/cs336/documents
/data2/conda/anaconda3/envs/vllm/bin/python


In [161]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # documents 的上一级

from cs336_basics.model import BasicsTransformerLM

# vocab_size = 32000
# context_length = 1024#4096
# d_model = 1024#4096
# num_layers = 7#32
# num_heads = 16#32
# d_ff = 4096#11008
# m_staff = BasicsTransformerLM(
#     vocab_size=vocab_size,
#     context_length=context_length,
#     d_model=d_model,
#     d_ff=d_ff,
#     num_heads=num_heads,
#     num_layers=num_layers
#     )

#for k, v in m_staff.state_dict().items():
    #print(k, tuple(v.shape))

> load_state_dict

In [169]:
# m = BasicsTransformerLM_(
#     vocab_size=vocab_size,
#     context_length=context_length,
#     d_model=d_model,
#     d_ff=d_ff,
#     num_heads=num_heads,
#     num_layers=num_layers
# )
key_map = {
    "token_embeddings": "embedding",
    "lm_head": "output_proj",
    "ln_final": "RMSNorm",
    ".attn.q_proj": ".CasalMutiheadAttn.proj_q",
    ".attn.k_proj": ".CasalMutiheadAttn.proj_k",
    ".attn.v_proj": ".CasalMutiheadAttn.proj_v",
    ".attn.output_proj": ".CasalMutiheadAttn.proj_out",
    ".ffn.": ".SwiGLU.",
    ".ln1.": ".RMSNorm1.",
    ".ln2.": ".RMSNorm2.",
}

def convert_sd(sd):
    out = {}
    for k, v in sd.items():
        for old, new in key_map.items():
            k = k.replace(old, new)
        out[k] = v
    return out

def map_key(k: str) -> str:
    for old, new in key_map.items():
        k = k.replace(old, new)
    return k

# m.load_state_dict(convert_sd(m_staff.state_dict()), strict=True)


- 参数量与 meta 设备一致

In [ ]:
# 未见，补一行 sum(p.numel() ...) 即可
# todo

>辅助函数

In [170]:
def log_softmax(x, dim=-1):
    x_max = torch.max(x, dim=dim, keepdim=True)[0]
    x = x - x_max
    return x - torch.log(torch.sum(torch.exp(x), dim=dim, keepdim=True))


def cross_entropy(inputs, targets):
    negative_log_softmax_logits = -log_softmax(inputs)
    return torch.mean(torch.gather(negative_log_softmax_logits, -1, targets.unsqueeze(-1)))

def rel(a, b):
    """相对误差 ||a-b||/||b||，在 fp64 中计算，避免量尺自身误差"""
    a, b = a.double(), b.double()
    return ((a - b).norm() / b.norm().clamp_min(1e-12)).item()


- fp32 的 logits 与梯度一致

- 初始 loss 约 $\ln V+\sigma^2/2$

In [171]:
cfg = dict(vocab_size=10000, context_length=256, d_model=256,
           num_layers=4, num_heads=4, d_ff=1024, rope_theta=10000.0)
staff = BasicsTransformerLM(**cfg).cuda().eval()
mini  = BasicsTransformerLM_(**cfg).cuda().eval()
mini.load_state_dict(convert_sd(staff.state_dict()), strict=True)

x = torch.randint(0, cfg["vocab_size"], (2, 128), device="cuda")
y = torch.randint(0, cfg["vocab_size"], (2, 128), device="cuda")
V = cfg["vocab_size"]

def grads(model):
    model.zero_grad(set_to_none=True)
    loss = cross_entropy(model(x).view(-1, V), y.view(-1)); loss.backward()
    return loss.item(), {n: p.grad.detach().clone() for n, p in model.named_parameters()}

la, ga = grads(staff); lb, gb = grads(mini)
print("loss", la, lb)
print("logits rel", rel(staff(x), mini(x)))
for n, g_staff in ga.items():
    k = map_key(n)                 # staff 参数名 → 你的参数名
    if k not in gb:
        print("未映射的 key:", n, "→", k)   # 兜底提示，别静默跳过
        continue
    r = rel(gb[k], g_staff)
    if r > 1e-4:
        print("GRAD MISMATCH", n, "→", k, f"{r:.3e}")


loss 9.231874465942383 9.231874465942383
logits rel 0.0


In [172]:
# 梯度循环"沉默"不等于通过：先确认它真的比较了东西
print("compared grads:", len(ga), "mapped:", sum(map_key(n) in gb for n in ga))

# 破坏性测试：扰动 mini 的一个参数，误差必须变大
with torch.no_grad():
    p = next(mini.parameters()); p.add_(1e-3 * torch.randn_like(p))
print("perturbed logits rel", rel(staff(x), mini(x)))   # 应明显 > 0
mini.load_state_dict(convert_sd(staff.state_dict()), strict=True)  # 恢复
print("restored", rel(staff(x), mini(x)))                # 回到 0.0

worst = max((rel(gb[map_key(n)], g), n) for n, g in ga.items())
print("worst grad rel:", worst)

compared grads: 39 mapped: 39
perturbed logits rel 0.0012840677465067658
restored 0.0
worst grad rel: (0.0, 'token_embeddings.weight')


- 误差预算与 bf16（10.4）

因为 fp32 下已经逐位相同，$e_{ms}=0$ 满足判据，所以这里主要是确认 mini 在 fp64 和 bf16 下也没有隐藏的 dtype 问题（比如 RoPE 缓存硬编码成 fp32）。

In [173]:
from contextlib import contextmanager
@contextmanager
def default_dtype(dt):
    old = torch.get_default_dtype(); torch.set_default_dtype(dt)
    try: yield
    finally: torch.set_default_dtype(old)

with default_dtype(torch.float64):
    ref   = BasicsTransformerLM(**cfg).cuda().eval()
    mini64 = BasicsTransformerLM_(**cfg).cuda().eval()
ref.load_state_dict({k: v.double() for k, v in staff.state_dict().items()})
mini64.load_state_dict(convert_sd(ref.state_dict()), strict=True)

with torch.no_grad():
    r = ref(x)
    e_s  = rel(staff(x), r)
    e_m  = rel(mini(x), r)
    e_ms = rel(mini(x), staff(x))
    print(e_s, e_m, e_ms)                      # 预期 1e-6 ~ 1e-5，且 e_ms <= max(e_s, e_m)
    print("fp64 mini vs ref", rel(mini64(x), r))   # 预期 < 1e-12

    with torch.autocast("cuda", dtype=torch.bfloat16):
        print("bf16 rel", rel(mini(x), staff(x)))   # 预期 1e-3 ~ 1e-2

4.1000320338951076e-07 4.1000320338951076e-07 0.0
fp64 mini vs ref 0.0
bf16 rel 0.0


In [175]:
with torch.no_grad():
    o32 = mini(x)
    with torch.autocast("cuda", dtype=torch.bfloat16):
        o16 = mini(x)
    print(o16.dtype, "bf16 vs fp32:", rel(o16, o32))   # 预期 1e-3 ~ 1e-2

torch.bfloat16 bf16 vs fp32: 0.005425869146535595


- 步骤 B：语义检验

In [176]:
# 因果性：改动位置 t 之后的 token，位置 <= t 的输出不能变
x2 = x.clone(); x2[:, 64:] = torch.randint(0, V, x2[:, 64:].shape, device="cuda")
with torch.no_grad():
    print("causal", rel(mini(x2)[:, :64], mini(x)[:, :64]))   # 应为 0 或 ~1e-7

causal 0.0


- 步骤 C：训练级校验（10.6）

轨迹对拍：两个模型从同一份权重出发，用同一串 batch 和 staff 的 AdamW 各训练 50 步，比较 loss 与参数。


In [177]:
import copy
def make_pair():
    s = BasicsTransformerLM(**cfg).cuda().train()
    m = BasicsTransformerLM_(**cfg).cuda().train()
    m.load_state_dict(convert_sd(s.state_dict()), strict=True)
    return s, m

s, m = make_pair()
opt_s = torch.optim.AdamW(s.parameters(), lr=1e-3, weight_decay=0.01)
opt_m = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=0.01)

g = torch.Generator(device="cuda").manual_seed(0)
for step in range(50):
    xb = torch.randint(0, V, (8, 128), device="cuda", generator=g)
    yb = torch.randint(0, V, (8, 128), device="cuda", generator=g)
    losses = []
    for model, opt in ((s, opt_s), (m, opt_m)):
        opt.zero_grad(set_to_none=True)
        loss = cross_entropy(model(xb).view(-1, V), yb.view(-1))
        loss.backward(); opt.step()
        losses.append(loss.item())
    if step < 5 or step % 10 == 9:
        print(step, losses, abs(losses[0]-losses[1]) / losses[0])

sd_m = m.state_dict()
worst = max((rel(sd_m[map_key(k)], v), k) for k, v in s.state_dict().items())
print("param rel after 50 steps:", worst)

0 [9.227084159851074, 9.227084159851074] 0.0
1 [9.238571166992188, 9.238571166992188] 0.0
2 [9.23733901977539, 9.23733901977539] 0.0
3 [9.230413436889648, 9.230413436889648] 0.0
4 [9.229707717895508, 9.229707717895508] 0.0
9 [9.227823257446289, 9.227823257446289] 0.0
19 [9.238089561462402, 9.238089561462402] 0.0
29 [9.248035430908203, 9.248035430908203] 0.0
39 [9.23974609375, 9.23974609375] 0.0
49 [9.246718406677246, 9.246718406677246] 0.0
param rel after 50 steps: (0.0, 'token_embeddings.weight')


单 batch 过拟合：固定 $B=4,T=64$，训 100 到 300 步，loss 应单调降到接近 0。

In [178]:
_, m = make_pair()
opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
xb = torch.randint(0, V, (4, 64), device="cuda")
yb = torch.randint(0, V, (4, 64), device="cuda")
for step in range(300):
    opt.zero_grad(set_to_none=True)
    loss = cross_entropy(m(xb).view(-1, V), yb.view(-1)); loss.backward(); opt.step()
    if step % 30 == 0: print(step, loss.item())

0 9.20885944366455
30 0.0018963555339723825
60 0.0005832839524373412
90 0.0004956793854944408
120 0.0004550408630166203
150 0.00042154808761551976
180 0.0003922139876522124
210 0.00036633521085605025
240 0.0003433715901337564
270 0.0003228624409530312



- 步骤 D：HF checkpoint（10.7）

先 AutoConfig 核对架构检查表，再下载权重、打印键名、写 convert_llama_to_staff，然后设置 eps，最后比较 logits、逐层和贪心生成。

要点：反例检验（去掉 q_proj 置换，误差应变成 O(1)）。

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from transformers import AutoConfig   # ← 必须在设置环境变量之后再 import
c = AutoConfig.from_pretrained("JackFram/llama-160m")
print(c.model_type, c.rope_scaling, c.num_key_value_heads,
      c.num_attention_heads, c.tie_word_embeddings, c.rms_norm_eps)


/data2/conda/anaconda3/envs/vllm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


llama {'rope_theta': 10000.0, 'rope_type': 'default'} 12 12 False 1e-06


- 步骤 E：存金标准（10.8）

用 10.8 的代码存 results/golden/small_fp32.pt，bf16 另存一份，之后改 FlashAttention 或 torch.compile 时直接回归比对。

### 11. 代价核算

#### 11.1 目的

把第 2 到 9 章的局部估算合并成一张总表，并产出两样东西：训练一步的 FLOPs 拆分，和一份"预测"，供第 12 章 profile 之后逐项对照。

#### 11.2 模块级核算总表

符号同第 1 章，$N=B\cdot T$，$d_{ff}=4d$。

| 模块（出现次数） | 参数 | FLOPs（forward） | 为反向保存的张量（元素数） | 类型 | kernel 数（forward，估） |
|---|---|---|---|---|---|
| Embedding（1） | $Vd$ | 0 | 索引 $N$ | 访存 | 1 |
| RMSNorm（$2L+1$） | $d$ | 约 $5Nd$ | $Nd$ 加 $N$ | 访存 | 约 6 |
| $q,k,v,o$ 投影（$4L$） | $d^2$ 各 | $2Nd^2$ 各 | 输入 $Nd$ | 计算 | 1 各 |
| RoPE（$2L$ 次调用） | 0 | 约 $3Nd$ 每次 | 中间量约 $Nd$ | 访存 | 约 7 每次 |
| $QK^\top$（$L$） | 0 | $2BT^2d$ | $Q,K$ | 计算，强度偏低 | 1 |
| 缩放、mask、softmax（$L$） | 0 | $O(BhT^2)$ | $c\cdot BhT^2$ | 访存 | 约 7 |
| $PV$（$L$） | 0 | $2BT^2d$ | $P,V$ | 计算，强度偏低 | 1 |
| SwiGLU 三个 GEMM（$L$） | $3dd_{ff}$ | $6Ndd_{ff}$ | $5Nd_{ff}$ 加 $Nd$ | 计算 | 3 |
| SwiGLU 逐元素（$L$） | 0 | $O(Nd_{ff})$ | 含在上一行 | 访存 | 3 |
| 残差加法（$2L$） | 0 | $Nd$ 各 | 无 | 访存 | 1 各 |
| `lm_head`（1） | $Vd$ | $2NdV$ | $Nd$ | 计算 | 1 |
| 交叉熵（1） | 0 | $O(NV)$ | $NV$ 的若干份 | 访存 | 数个 |

数字的出处：第 4 到 9 章。"kernel 数"和"保存份数"是估计，第 12 章实测后替换。

#### 11.3 训练一步的 FLOPs 拆分

**forward**（每个 token）：

$$
\underbrace{8d^2L}_{\text{注意力投影}}+\underbrace{24d^2L}_{\text{FFN}}+\underbrace{4TdL}_{\text{注意力 matmul}}+\underbrace{2dV}_{\texttt{lm\_head}}
$$

训练一步为 forward 的 3 倍（backward 约 2 倍，见第 6、7 章）。不含 optimizer，也不含逐元素运算。

```python
def flops_breakdown(d, L, V, B, T, d_ff=None):
    d_ff = d_ff or 4 * d; N = B * T
    f = {"attn_proj": L * 8 * N * d * d, "ffn": L * 6 * N * d * d_ff,
         "attn_matmul": L * 4 * N * T * d, "lm_head": 2 * N * d * V}
    tot = sum(f.values())
    return {**f, "fwd_total": tot, "step_total": 3 * tot}
```

**small，$B=4$，$T=512$**：

| 项 | forward (GFLOP) | 占比 |
|---|---|---|
| 注意力投影 | 115.9 | 21.7% |
| FFN | 347.9 | 65.2% |
| 注意力 matmul | 38.7 | 7.2% |
| `lm_head` | 31.5 | 5.9% |
| 合计 | 534.0 | 100% |

训练一步约 1.60 TFLOP，与 Step 2 一致。用 `flops_breakdown` 画一张堆叠柱状图（横轴 $T$，纵轴占比）。

#### 11.4 为什么 ctx 增大时注意力占比上升

线性项（投影、FFN、`lm_head`）与 $N$ 成正比；注意力 matmul 与 $N\cdot T$ 成正比。二者之比（$d_{ff}=4d$）：

$$
\frac{\text{注意力 matmul}}{\text{层内线性}}=\frac{4NTd}{32Nd^2}=\frac{T}{8d}
$$

**注意力 matmul 的 FLOPs 占 forward 的比例**（$B$ 不影响占比）：

| 模型 | $T=128$ | $T=512$ | $T=1024$ | $T=4096$ |
|---|---|---|---|---|
| small（$d=768$） | 1.9% | 7.2% | 13.5% | 38.4% |
| large（$d=1280$） | 1.2% | 4.7% | 9.0% | 28.3% |

宽度越大，占比越低（分母里的 $d^2$ 比分子的 $d$ 增长更快）。

**FLOPs 占比不等于时间占比**。注意力里的缩放、mask、softmax 是 $T^2$ 大小的访存密集运算（第 6 章），按我在第 8 章的估算，small 在 $T=512$ 时注意力核心约占一个 block 时间的三分之一，而 FLOPs 只占约 7%。把这两行并排写进表里，是本章最有价值的一个结论，待 nsys 验证：

| 项 | FLOPs 占比 | 预测时间占比 | 实测时间占比 |
|---|---|---|---|
| 注意力（matmul + 缩放 + mask + softmax） | 7.2% | 约 35%（第 8 章） | |
| FFN（GEMM + 逐元素） | 65.2% | 约 40% | |
| 其余 | | | |

#### 11.5 预测 → 回填

| 阶段 | FLOPs | $P$ | 理想时间 | 访存部分 | 预测总时间 | 实测 | 比值 |
|---|---|---|---|---|---|---|---|
| forward（small，$T=512$） | 0.534 T | 33.4 | 16.0 ms | 约 20 ms | 约 37 ms | | |
| backward | 1.068 T | 33.4 | 32.0 ms | 待估 | | | |
| optimizer | 28 B/参数 | BW 769 GB/s | 4.7 ms（下限） | | | | |

backward 的访存部分可以先取 forward 的约 2 倍作为粗估，实测后修正。

#### 11.6 显存拆分

| 部分 | 估算 | 验证方法 |
|---|---|---|
| 权重 + 梯度 + Adam | $16P$ B（fp32） | `max_memory_allocated`（optimizer step 后） |
| FFN 激活 | $5Nd_{ff}$ 每层 | `saved_tensors_hooks`（第 8 章） |
| 注意力分数 | $c\,BhT^2$ 每层，$c\approx1$ 到 3 | 同上 |
| 其余 $N\times d$ 张量 | 约 15 到 25 份每层 | 同上 |
| logits 与损失 | $NV$ 的约 3 到 4 份 | 同上 |

写一个函数输出预估，与 `torch.cuda.max_memory_allocated()` 比较：

```python
def est_peak_gib(P, d, L, h, V, B, T, c=2, other_per_layer=20):
    N = B * T
    static = 16 * P
    act = L * 4 * (5 * N * 4 * d + c * B * h * T * T + other_per_layer * N * d) + 4 * 4 * N * V
    return (static + act) / 2**30
```

两边的差值就是需要在注解里解释的内容（份数估错、临时 buffer、allocator 碎片）。

### 12. NVTX 标注与 Profile 教程

#### 12.1 方法：预测 → 测量 → 解释

Part 1 的 profile 不是为了"看一眼哪里慢"，而是闭合下面这个循环：

1. 第 6、8、9、11 章已经给出**预测**（kernel 数、各部分时间占比）；
2. 用不加 profiler 的 `benchmark.py` 得到**真实耗时**（nsys 有开销，不能用来量绝对时间）；
3. 用 nsys 得到**构成**（kernel 种类、次数、占比）；
4. 对照预测，**解释**差异。

分工：`benchmark.py` 回答"多快"，nsys 回答"为什么是这个速度"。

#### 12.2 NVTX 基础

**它是什么**：在 CPU 侧给一段代码打上带名字的区间，nsys 在时间线上显示出来。GPU 是异步的，所以：

| 现象 | 说明 |
|---|---|
| 区间里的 PyTorch 调用只是**发射** kernel | 若区间结束时没有 `synchronize`，CPU 区间很短，GPU 工作还在后面 |
| nsys 会把 kernel **投影**回发射它的 NVTX 区间 | 这才是"该区间对应的 GPU 工作" |
| 投影得到的是 kernel 的**时间跨度**（含 kernel 之间的空隙），不是 kernel 耗时之和 | 想要耗时之和，用按区间汇总 kernel 的报告（见 12.6） |

**API**（`torch.cuda.nvtx`）：

```python
import torch.cuda.nvtx as nvtx
with nvtx.range("forward"):       # 推荐，自动配对
    ...
nvtx.range_push("x"); ...; nvtx.range_pop()   # 手动配对，跨函数时用
nvtx.mark("checkpoint")                        # 时间线上的一个瞬时标记
```

**命名规范**：
- 用**静态字符串**。nsys 按名字汇总，名字里带 step 编号会让每个 step 变成一行；
- 层级用点号：`attn.scores`、`attn.softmax`、`attn.av`；
- 同一类层用**同一个名字**（如 `block`），汇总时调用次数就是层数，便于核对"每个 forward 应有 $L$ 次"；
- 需要看单个层时，临时加编号，不要作为默认。

**开销**：每个区间是微秒量级的 CPU 开销。标注到每个小算子会让 CPU 变慢，对小模型、短 ctx（本来就是发射受限）影响尤其明显。所以**粗粒度标注用于计时，细粒度标注只在 profile 运行里开**。

#### 12.3 本项目的标注方案

| 区间 | 内容 | 位置 |
|---|---|---|
| `step` | 一次完整训练步 | 最外层 |
| `forward` | embedding 到 logits | 主线程 |
| `loss` | 交叉熵 | 主线程 |
| `backward` | `loss.backward()` | 见 12.7，**归因有特殊问题** |
| `optimizer` | `opt.step()` 与 `zero_grad` | 主线程 |
| `embed`、`block`、`ln_final`、`lm_head` | forward 内部 | forward hook |
| `ln1`、`attn`、`ln2`、`ffn` | block 内部 | forward hook |
| `attn.scores`、`attn.softmax`、`attn.av` | attention 内部三段 | 替换函数（作业要求） |

**attention 内部三段**：把 staff 的 `scaled_dot_product_attention` 换成带标注的版本。`CausalMultiHeadSelfAttention.forward` 在调用时按模块全局名查找这个函数，所以替换模块属性即可，不需要改 staff 源码：

```python
import math, torch, torch.cuda.nvtx as nvtx
from einops import einsum
import cs336_basics.model as M
from cs336_basics.nn_utils import softmax

def annotated_sdpa(Q, K, V, mask=None):
    d_k = K.shape[-1]
    with nvtx.range("attn.scores"):                     # QK^T、缩放、mask
        s = einsum(Q, K, "... q d, ... k d -> ... q k") / math.sqrt(d_k)
        if mask is not None:
            s = torch.where(mask, s, float("-inf"))
    with nvtx.range("attn.softmax"):
        w = softmax(s, dim=-1)
    with nvtx.range("attn.av"):
        return einsum(w, V, "... q k, ... k d -> ... q d")

M.scaled_dot_product_attention = annotated_sdpa          # 只在 profile 运行里替换
```

**模块级标注**（用 hook，不改源码）：

```python
def add_nvtx_hooks(model, names=("RMSNorm", "CausalMultiHeadSelfAttention", "SwiGLU",
                                 "TransformerBlock", "Embedding", "Linear")):
    hs = []
    for mod in model.modules():
        k = type(mod).__name__
        if k in names:
            hs.append(mod.register_forward_pre_hook(lambda m, a, k=k: nvtx.range_push(k)))
            hs.append(mod.register_forward_hook(lambda m, a, o: nvtx.range_pop()))
    return hs
```

这里用类名作为区间名，所以所有 `RMSNorm` 汇总成一行，调用次数应为 $2L+1$，是个很好的自检（`Linear` 包含 `lm_head`，数量是 $7L+1$）。

注意：forward hook 只覆盖 forward，不覆盖反向。

#### 12.4 profile 运行脚本（骨架）`scripts/profile_run.py`

```python
def phase(name, fn):
    with nvtx.range(name):
        out = fn()
        torch.cuda.synchronize()      # profile 模式下每个阶段末尾同步，GPU 工作才会落在对应区间的时间窗内
    return out

# 预热，不被抓取
for _ in range(args.warmup): train_step()
torch.cuda.synchronize()

torch.cuda.profiler.start()           # 只在这里开始抓取
for _ in range(args.steps):           # 3 步足够，多了文件很大
    with nvtx.range("step"):
        loss = phase("forward", fwd_and_loss)
        phase("backward", loss.backward)
        phase("optimizer", opt_step_and_zero)
torch.cuda.synchronize()
torch.cuda.profiler.stop()
```

参数：`--model --ctx --batch --dtype --mode {fwd,fwdbwd,full} --warmup --steps --nvtx-fine`。`--nvtx-fine` 打开 12.3 里的替换与 hook。

**每个阶段末尾同步会破坏阶段之间的重叠**，所以 profile 运行得到的是"各阶段互不重叠"的构成，而不是真实的流水线。这对归因是好事，对测速不是，所以 12.1 里强调二者分开。

#### 12.5 运行 nsys

```bash
cd /data2/c220/hz/cs336
mkdir -p results/nsys
CUDA_VISIBLE_DEVICES=0 uv run nsys profile \
  -o results/nsys/small_ctx512_fp32 --force-overwrite=true \
  --trace=cuda,nvtx,osrt \
  --capture-range=cudaProfilerApi --capture-range-end=stop \
  python scripts/profile_run.py --model small --ctx 512 --batch 4 --mode full \
         --warmup 5 --steps 3 --nvtx-fine
```

| 选项 | 作用 |
|---|---|
| `--trace=cuda,nvtx,osrt` | 抓 CUDA API 与 kernel、NVTX、OS 运行时（沿用你的服务器指南） |
| `--capture-range=cudaProfilerApi` 加 `--capture-range-end=stop` | **只抓 `profiler.start()` 到 `stop()` 之间**，预热不进文件 |
| `--force-overwrite=true` | 覆盖同名文件 |
| `--cuda-memory-usage=true`（可选） | 记录显存分配时间线，有额外开销 |

输出命名要带配置（模型、ctx、dtype），否则很快分不清。

**开始前**：`nvidia-smi` 确认卡是空的；另一张卡用 `CUDA_VISIBLE_DEVICES` 隔离。

#### 12.6 读结果

**命令行（在服务器上）**。报告名随版本变化，先 `nsys stats --help-reports` 看本机有哪些：

| 想看什么 | 报告（常见名字） |
|---|---|
| 全部 kernel 的汇总（次数、总时间、平均） | `cuda_gpu_kern_sum` |
| 每个 NVTX 区间的 CPU 时间 | `nvtx_sum` |
| 每个 NVTX 区间投影到 GPU 的**时间跨度** | `nvtx_gpu_proj_sum` |
| **按 NVTX 区间汇总 kernel**（耗时之和） | `nvtx_kern_sum` |
| 拷贝（H2D、D2H、D2D）时间与大小 | `cuda_gpu_mem_time_sum`、`cuda_gpu_mem_size_sum` |
| CPU 侧 CUDA API 耗时（发射开销） | `cuda_api_sum` |
| 逐次 kernel 的明细 | `cuda_gpu_trace` |

```bash
uv run nsys stats --report cuda_gpu_kern_sum --report nvtx_sum \
     --format csv --output results/nsys/small_ctx512 results/nsys/small_ctx512_fp32.nsys-rep
```

CSV 的文件名和列名以实际输出为准，用 `glob` 找文件，用 `df.columns` 看列。

**汇总成"类别表"**（把 kernel 名归类，供 11.4 与 12.11 回填）：

```python
import re, pandas as pd
CATS = [("gemm", r"gemm|xmma|cutlass|cublas|sgemm"), ("softmax", r"softmax"),
        ("reduce", r"reduce"), ("elementwise", r"elementwise|vectorized"),
        ("copy/cat", r"copy|cat|memcpy"), ("index/embed", r"index|embedding|scatter|gather"),
        ("optimizer", r"multi_tensor|adam")]
def cat(name):
    for c, pat in CATS:
        if re.search(pat, name, re.I): return c
    return "other"
df["cat"] = df["Name"].map(cat)
print(df.groupby("cat")[["Total Time (ns)", "Instances"]].sum().assign(
    share=lambda t: t["Total Time (ns)"] / t["Total Time (ns)"].sum()))
```

先把归为 `other` 的最大几项看一遍，补全规则。

**GPU 利用率自检**：`kernel 总时间 ÷ 步的实际时间`（来自 `cuda_gpu_kern_sum` 与 `nvtx_sum` 的 `step`）。

| 比值 | 含义 |
|---|---|
| 接近 1 | GPU 受限，优化 kernel 本身 |
| 明显低于 1（如小于 0.6） | **发射受限**，GPU 在等 CPU；时间线上 kernel 之间有明显空隙 |

**时间线（Nsight Systems GUI）**：把 `.nsys-rep` 下载到本地（版本要不低于生成它的 nsys，用 `nsys --version` 对照）。

| 行 | 看什么 |
|---|---|
| NVTX | 区间的层级与宽度，先在这里定位 `forward`、`backward` |
| CUDA HW → Kernels | kernel 是否首尾相接；空隙意味着发射受限 |
| 主线程 → CUDA API | 一个 kernel 对应的发射调用，点击 kernel 可以跳到它 |
| 另一个线程 | autograd 的反向线程（见 12.7） |

操作：先缩放到一个 `block`，数 kernel 个数（对照 8.9 的约 50），再看哪几个最宽。

#### 12.7 backward 与 optimizer 的归因

**forward 与 optimizer** 都在主线程发射，NVTX 投影正常。

**backward 的问题**：PyTorch 的反向在 autograd 引擎的**工作线程**上发射 kernel，而 `backward` 区间是在主线程里打的（主线程只是阻塞等待）。nsys 的 NVTX 投影按发射线程匹配区间，所以**很可能**出现 `backward` 区间投影到的 kernel 很少或为 0。这是我基于机制的判断，不是对某个版本的实测，请按下面方法**先验证**：

1. 看 `nvtx_gpu_proj_sum` 或 `nvtx_kern_sum` 里 `backward` 的 kernel 数。若远少于 forward 的约 2 倍，就是这个问题。
2. 备选方案：

| 方案 | 做法 | 说明 |
|---|---|---|
| A. 时间窗归因 | 取 `backward` 区间的起止时间，从 `cuda_gpu_trace` 里选起始时间落在窗内的 kernel | 配合 12.4 的"每阶段末尾同步"，窗内的 GPU 工作就是该阶段的。最稳妥 |
| B. `emit_nvtx` | `with torch.autograd.profiler.emit_nvtx():` 包住整个 profile 区间 | 为每个 autograd 算子（含反向）自动打 NVTX，名字形如 `MmBackward0`，反向区间在引擎线程上，投影正确 |
| C. 新版 nsys 的 PyTorch 选项 | `nsys profile --help` 里查 `--pytorch` | 不确定你的版本是否支持 |

作业要求里"forward 与 forward+backward 中同一个 kernel 占比"这类问题，用方案 A 就足够。

#### 12.8 把 kernel 对应回代码

| 方法 | 做法 | 可靠度 |
|---|---|---|
| A. 单算子隔离 | 写一个小循环只跑一个算子（如 `torch.softmax`、一次 `einsum`），用 nsys 看它产生的 kernel 名 | **最可靠**，建议对 6 到 8 个关键算子各做一次，建立名字字典 |
| B. 细粒度 NVTX | 12.3 的区间，配合 `nvtx_kern_sum` | 可靠，缺点是 backward 归因问题 |
| C. `torch.profiler` | 给出 `aten::` 级别的 CUDA 时间，更接近代码层面（见 12.9） | 与 nsys 互补 |
| D. GUI 点击 | 点 kernel，跳到发射它的 CUDA API 调用与线程 | 适合个别 kernel |

**kernel 名速查**（名字依库版本与架构（sm_120）而变，**以你的实测为准**）：

| 类别 | 常见名字特征 | 对应代码 |
|---|---|---|
| GEMM | 含 `gemm`、`xmma`、`cutlass`、`cublas`、`sgemm` | `Linear`、$QK^\top$、$PV$、`lm_head` |
| 逐元素 | `elementwise`、`vectorized_elementwise_kernel` | RMSNorm 的乘除、RoPE、缩放、`where`、SiLU、残差 |
| 归约 | `reduce_kernel` | RMSNorm 的 `mean` |
| softmax | `SoftMax` | `softmax` 或其内部实现 |
| 拷贝 / cat | `copy`、`CatArrayBatchedCopy` | `.contiguous()`、`concat`、dtype 转换 |
| 索引 | `index`、`embedding`、`scatter` | embedding 的查表与反向 |
| 优化器 | `multi_tensor_apply`、或一大批小 kernel | AdamW |

**staff 的 AdamW 如果是逐参数的 Python 循环**，你会在 nsys 里看到数量极多的小 kernel，`cuda_api_sum` 里发射开销占主导。这是 11.5 里 optimizer 预测偏差的直接原因。

#### 12.9 配套工具

**`torch.profiler`**（算子级、代码级的视角）：

```python
from torch.profiler import profile, ProfilerActivity
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    train_step(); torch.cuda.synchronize()
print(prof.key_averages(group_by_input_shape=True).table(sort_by="cuda_time_total", row_limit=20))
prof.export_chrome_trace("results/trace.json")      # 可在 Perfetto 或 chrome://tracing 打开
```

新版本里排序键可能叫 `device_time_total`，按报错提示改。它给出 `aten::mm`、`aten::bmm`、`aten::softmax` 等算子的 CUDA 时间，适合回答"每类 PyTorch 算子占多少"。

**显存快照**（对应 3.8）：

```python
torch.cuda.memory._record_memory_history(max_entries=100000)
train_step(); torch.cuda.synchronize()
torch.cuda.memory._dump_snapshot("results/mem_small_512.pickle")
torch.cuda.memory._record_memory_history(enabled=None)
```

把 pickle 下载到本地，拖到 pytorch.org/memory_viz 查看（在浏览器里本地解析）。读法：
- "Active Memory Timeline"：峰值出现在哪个阶段（预期是 forward 末尾或 backward 开始）；
- 悬停到色块上看分配时的调用栈，识别 $T^2$ 大小的分数矩阵、$N\times d_{ff}$ 的 FFN 中间量；
- 与第 11.6 的预估对照。

辅助指标：`max_memory_allocated`（真实使用）与 `max_memory_reserved`（allocator 持有）之差反映碎片。

**分布式（Part 3 预告）**：nsys 对多进程可以用 `%q{RANK}` 在输出文件名里区分进程，每个 rank 各自调用 `profiler.start/stop`。到 Part 3 再展开。

#### 12.10 常见坑

| 坑 | 后果 | 处理 |
|---|---|---|
| 没有预热，或预热被抓进文件 | 看到 cuBLAS 初始化、懒加载，首次迭代的 kernel 不是稳态 | 用 capture range 只抓测量步 |
| 用 profile 运行的时间当作性能 | nsys 有开销，且每阶段同步 | 时间用 `benchmark.py`，nsys 只看构成 |
| NVTX 区间内没同步 | CPU 区间极短，被误读 | 阶段末尾 `synchronize`（仅 profile 模式） |
| backward 归因为空 | 误以为反向没有 kernel | 12.7 的验证与备选 |
| 步数太多 | 文件巨大，GUI 卡 | 3 步 |
| 把 `other` 类放着不管 | 占比失真 | 逐个看最大的 `other` 并补规则 |
| 标注粒度太细 | CPU 变慢，小模型尤其明显 | 计时不开细标注 |
| 版本不匹配（GUI 比 nsys 旧） | 打不开 | 用不低于 nsys 的 GUI |
| 卡不空闲 | 数据不可信 | 运行前 `nvidia-smi` |

#### 12.11 作业问题与取数方式对照（3.4 与 3.6）

| 问题 | 取数方式 |
|---|---|
| forward 中耗时最长的 kernel，调用次数 | `nvtx_kern_sum` 或时间窗法，取 `forward` 区间 |
| 同一个 kernel 在 forward 与 forward+backward 中的占比 | 两个区间各算一次占比 |
| 除 matmul 外占时间最多的 kernel | 类别表里除 `gemm` 之外最大的类别 |
| 完整训练步里 matmul 的占比 | 类别表（含 backward 与 optimizer） |
| attention 内 softmax 与 matmul 的时间比，对比 FLOPs 比 | `attn.scores`、`attn.softmax`、`attn.av` 三个区间 |
| 加了 `torch.compile` 之后变了什么 | 重新 profile，对比 kernel 数与类别表 |

#### 12.12 对照表（回填）

**类别级**（small，$B=4$，$T=512$，fp32，forward）：

| 类别 | 预测 kernel 数 | 实测 | 预测时间占比 | 实测占比 |
|---|---|---|---|---|
| gemm | 约 90（$7L+1$ 个线性层加 $2L$ 个注意力 matmul） | | 约 45% | |
| elementwise | 约 300 | | 约 40% | |
| reduce | 约 25 | | 约 3% | |
| softmax | 12 | | 约 5% | |
| copy / cat | 约 40 | | 约 3% | |
| other | | | | |

**区间级**（沿用 8.9 与 9.9 的预测）：

| 区间 | 预测 kernel 数 | 实测 | 预测时间 (ms) | 实测 (ms) | 占比 |
|---|---|---|---|---|---|
| embed | 1 | | 0.02 | | |
| block（×12） | 约 600 | | 34.8 | | |
| `attn.scores` | | | | | |
| `attn.softmax` | | | | | |
| `attn.av` | | | | | |
| ln_final | 6 | | 0.09 | | |
| lm_head | 1 | | 0.94 | | |
| loss | 数个 | | 1.1 | | |

类别表里的 90、300、25 等数字是我按 8.9 的 kernel 清单粗略相加的，误差较大，实测后直接替换。

### 13. 附录

#### 13.1 推导

**(a) softmax 加交叉熵的梯度**

对单个 token，logits $x\in\mathbb{R}^V$，目标类别 $y$：

$$
\ell=-\log p_y=-x_y+\mathrm{logsumexp}(x),\qquad p=\mathrm{softmax}(x)
$$

$$
\frac{\partial\ell}{\partial x_i}=-\mathbb{1}[i=y]+\frac{e^{x_i}}{\sum_je^{x_j}}=p_i-\mathbb{1}[i=y]
\quad\Rightarrow\quad \delta x=p-e_y
$$

对 $N$ 个 token 取平均，则 $\delta x=(P-E_Y)/N$。

**与第 6.3 节的雅可比互相验证**：先求 $\partial\ell/\partial p_j=-\mathbb{1}[j=y]/p_y$，再乘以 $J=\mathrm{diag}(p)-pp^\top$：

$$
\delta x_i=\sum_j\frac{\partial\ell}{\partial p_j}J_{ji}=-\frac{1}{p_y}\,p_y(\delta_{yi}-p_i)=p_i-\delta_{yi}
$$

结果一致。**融合后的梯度只需要 $p$ 与目标，不需要显式构造 $V\times V$ 的雅可比，也避免了 $1/p_y$ 这种在 $p_y\to0$ 时不稳定的中间量。**

**稳定计算**：$\mathrm{logsumexp}(x)=m+\log\sum_je^{x_j-m}$，$m=\max_jx_j$。直接对 softmax 的输出取对数，在概率下溢为 0 时会得到 $-\infty$。

**初始 loss**：$\ell\approx\ln V+\sigma_{logit}^2/2$（第 9.3 节）。

**(b) 其余推导的位置**

| 内容 | 章节 |
|---|---|
| RMSNorm 的反向与尺度不变性 | 4.2 |
| RoPE 的相对位置性质、反向为逆旋转 | 5.2、5.4 |
| softmax 的雅可比、注意力反向与 $D=\mathrm{rowsum}(\delta O\odot O)$ | 6.3、6.5 |
| SiLU 的导数、SwiGLU 反向 | 7.2、7.4 |
| 残差的梯度恒等路径 | 8.3 |
| Embedding 的 scatter-add 梯度 | 2.2 |

**(c) AdamW（Part 4 需要）**

$$
m\leftarrow\beta_1m+(1-\beta_1)g,\quad v\leftarrow\beta_2v+(1-\beta_2)g^2,\quad
\hat m=\tfrac{m}{1-\beta_1^t},\ \hat v=\tfrac{v}{1-\beta_2^t}
$$
$$
\theta\leftarrow\theta-\alpha\,\frac{\hat m}{\sqrt{\hat v}+\epsilon}-\alpha\lambda\theta
$$

- 权重衰减与梯度**解耦**（直接作用在 $\theta$ 上，不进入 $m,v$）；
- 状态是 $m$ 和 $v$，各一份与参数同形状的张量，所以 optimizer 状态是参数的 2 倍；
- 访存估算（Step 2）：读 $p,g,m,v$，写 $p,m,v$，约 28 B/参数（fp32）。

具体超参与实现细节以你手上的作业要求为准。

#### 13.2 自测题

| 章 | 问题 | 提示 |
|---|---|---|
| 2 | 为什么 embedding 的梯度张量是稠密的，而本 batch 只出现了少数词？ | 梯度张量的形状等于参数形状 |
| 3 | 为什么权重存成 `(d_out, d_in)`？ | 对应 $y=xW^\top$ 的行向量写法 |
| 4 | RMSNorm 的 $\delta x$ 为什么与 $x$ 正交？ | 尺度不变性 |
| 4 | bf16 下为什么不能直接对 $x^2$ 求和？ | 尾数位数与累加误差 |
| 5 | 为什么 RoPE 之后的 $QK^\top$ 只依赖相对位置？ | $R(m\theta)^\top R(n\theta)=R((n-m)\theta)$ |
| 5 | 为什么 staff 的输出布局与教科书不同却不影响结果？ | 同时置换 Q 与 K 的分量，点积不变 |
| 6 | 为什么除以 $\sqrt{d_k}$ 而不是 $d_k$？ | 方差是 $d_k$，标准差是 $\sqrt{d_k}$ |
| 6 | $T$ 加倍，注意力显存、FLOPs 各变为几倍？ | 都是 4 倍 |
| 6 | 为什么 $D=\mathrm{rowsum}(\delta O\odot O)$ 能代替对 $P$ 的归约？ | 代入 $\delta P_{ij}=\delta o_i\cdot v_j$ |
| 7 | SwiGLU 的 `w1` 与 `w3` 能互换吗？ | 一个过 SiLU，一个不过，角色不对称 |
| 7 | 为什么本作业的 FFN 参数是 GPT-2 的 1.5 倍？ | 三个矩阵对两个矩阵 |
| 8 | 子层输出归零时，pre-norm 的 block 为什么是恒等映射？ | $y=x+F(\cdot)$，$F=0$ |
| 8 | pre-norm 为什么需要 `ln_final`？ | 主干没有归一化 |
| 9 | 绑定 embedding 与 `lm_head` 节省什么，不节省什么？ | 节省参数与显存，不节省 FLOPs |
| 9 | 为什么 small 的参数里 embedding 与 head 占比高？ | $V$ 固定，$d$ 小 |
| 10 | 用 staff 作为真值的对拍，不能发现哪类错误？ | 与 staff 共有的错误 |
| 10 | 为什么对拍前必须关 TF32？ | 引入约 $10^{-3}$ 的误差 |
| 11 | 为什么注意力的时间占比比 FLOPs 占比高得多？ | 访存密集的 $T^2$ 逐元素运算 |
| 11 | backward 为什么约是 forward 的 2 倍？ | 对输入和对权重各一次同规模 matmul |
| 12 | 为什么 NVTX 区间内不同步就测不准？ | GPU 异步，区间只覆盖发射 |
| 12 | 为什么用 nsys 的时间当性能数据不对？ | profiler 开销、阶段同步 |

#### 13.3 参考文献

| 主题 | 文献 |
|---|---|
| Transformer | Vaswani et al., 2017, *Attention Is All You Need* |
| RoPE | Su et al., 2021, *RoFormer: Enhanced Transformer with Rotary Position Embedding* |
| RMSNorm | Zhang & Sennrich, 2019, *Root Mean Square Layer Normalization* |
| LayerNorm | Ba et al., 2016, *Layer Normalization* |
| GLU 变体 | Shazeer, 2020, *GLU Variants Improve Transformer* |
| SiLU / Swish | Ramachandran et al., 2017, *Searching for Activation Functions* |
| pre-norm 与 post-norm | Xiong et al., 2020, *On Layer Normalization in the Transformer Architecture* |
| LLaMA（架构参照） | Touvron et al., 2023, *LLaMA: Open and Efficient Foundation Language Models* |
| 权重绑定 | Press & Wolf, 2016, *Using the Output Embedding to Improve Language Models* |
| 混合精度 | Micikevicius et al., 2017, *Mixed Precision Training* |
| FlashAttention | Dao et al., 2022（FlashAttention）；Dao, 2023（FlashAttention-2） |
| Roofline | Williams, Waterman, Patterson, 2009, *Roofline: An Insightful Visual Performance Model* |
| ZeRO（Part 4） | Rajbhandari et al., 2020, *ZeRO: Memory Optimizations Toward Training Trillion Parameter Models* |
| 工具文档 | PyTorch 文档中 `torch.cuda.nvtx`、`torch.profiler`、CUDA memory snapshot；NVIDIA Nsight Systems User Guide |

## 使用建议

- **做完第 3、5、6 章再决定要不要继续**：这三章最能帮你看清 matmul、softmax、elementwise 在哪里，是 nsys 归因真正需要的。
- **先写 `MiniTransformerLM` 的最小版本（第 9、10 章先通），再回头补推导**。先通过对拍，后面的数学注解才有可验证的落点。
- **做完后把核心类抽到 `cs336_systems/mini_model.py`**，让 `benchmark.py` 可以通过一个开关在 staff 模型和 mini 模型之间切换，这本身就是一个很好的交叉验证。
- Jupyter 在远程服务器上用端口转发：服务器端 `uv run jupyter lab --no-browser --port 8888`，本地 `ssh -L 8888:localhost:8888 <user>@c220`。